### # AI Resume Screening & Job Recommendation System

## 01 — NLP Fundamentals

This notebook covers the basic NLP techniques required for
resume screening and job recommendation.

### Topics
- Natural Language Processing
- Text preprocessing
- Bag of Words
- TF-IDF
- Cosine Similarity
- Resume ↔ Job Description similarity

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.preprocessing import preprocess_text

In [3]:
from src.resume_parser import extract_resume_text

#### 1. Understanding Text Data

Before applying machine learning, we need to understand how
computers represent human language.

We will begin with a simple sentence.

In [4]:
text = "Python is powerful for machine learning"

print(text)

Python is powerful for machine learning


#### 2. Bag of Words

Bag of Words (BoW) represents text using the frequency
of words appearing in each document.

It ignores grammar and word order and focuses mainly on
which words appear and how often they appear.

In [5]:
documents = [
    "Python machine learning",
    "Python data science"
]

vectorizer = CountVectorizer()

bow_matrix = vectorizer.fit_transform(documents)

print("Vocabulary:")
print(vectorizer.get_feature_names_out())

print("\nBag of Words Matrix:")
print(bow_matrix.toarray())

Vocabulary:
['data' 'learning' 'machine' 'python' 'science']

Bag of Words Matrix:
[[0 1 1 1 0]
 [1 0 0 1 1]]


#### 3. TF-IDF

TF-IDF stands for Term Frequency-Inverse Document Frequency.

Unlike Bag of Words, TF-IDF does not simply count how many
times a word appears. It gives more importance to words that
are useful for distinguishing one document from another.

In [6]:
tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("Vocabulary:")
print(tfidf_vectorizer.get_feature_names_out())

print("\nTF-IDF Matrix:")
print(tfidf_matrix.toarray())

Vocabulary:
['data' 'learning' 'machine' 'python' 'science']

TF-IDF Matrix:
[[0.         0.6316672  0.6316672  0.44943642 0.        ]
 [0.6316672  0.         0.         0.44943642 0.6316672 ]]


#### 4. Resume ↔ Job Description Similarity

We will use TF-IDF to represent a resume and a job description
as numerical vectors, then use cosine similarity to measure
how closely they match.

In [7]:
## resume and job description
resume = """
Python developer with experience in machine learning,
pandas, NumPy, SQL and scikit-learn.
"""

job_description = """
Looking for a Python developer with experience in React, JavaScript, HTML, CSS and TypeScript
"""

In [8]:
##Convert both texts into TF-IDF vectors
documents = [resume, job_description]

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Vocabulary:")
print(tfidf_vectorizer.get_feature_names_out())
print("\nTF-IDF Matrix:")
print(tfidf_matrix.toarray())

TF-IDF Matrix Shape: (2, 20)
Vocabulary:
['and' 'css' 'developer' 'experience' 'for' 'html' 'in' 'javascript'
 'learn' 'learning' 'looking' 'machine' 'numpy' 'pandas' 'python' 'react'
 'scikit' 'sql' 'typescript' 'with']

TF-IDF Matrix:
[[0.22457838 0.         0.22457838 0.22457838 0.         0.
  0.22457838 0.         0.31563707 0.31563707 0.         0.31563707
  0.31563707 0.31563707 0.22457838 0.         0.31563707 0.31563707
  0.         0.22457838]
 [0.22457838 0.31563707 0.22457838 0.22457838 0.31563707 0.31563707
  0.22457838 0.31563707 0.         0.         0.31563707 0.
  0.         0.         0.22457838 0.31563707 0.         0.
  0.31563707 0.22457838]]


In [9]:
## calculate cosine similarity between the two vectors
similarity = cosine_similarity(
    tfidf_matrix[0],
    tfidf_matrix[1]
)

print("Similarity Matrix:")
print(similarity)

Similarity Matrix:
[[0.30261268]]


In [10]:
score = similarity[0][0]

print(f"Resume-Job Match Score: {score:.2%}")

Resume-Job Match Score: 30.26%


## 5. Text Preprocessing

Text preprocessing prepares raw text for NLP by removing
noise and standardizing the content.

#### 5.1 Lowercasing

Words like "Python", "PYTHON", and "python" should be treated
as the same token.

In [11]:
raw_text = """
John DOE
Python Developer
Email: john123@gmail.com

Skills: Python, SQL, Pandas, NumPy!!!
"""

print("Original Text:\n")
print(raw_text)

clean_text = raw_text.lower()

print("\nAfter Lowercase:\n")
print(clean_text)

Original Text:


John DOE
Python Developer
Email: john123@gmail.com

Skills: Python, SQL, Pandas, NumPy!!!


After Lowercase:


john doe
python developer
email: john123@gmail.com

skills: python, sql, pandas, numpy!!!



#### 5.2 Removing Punctuation

Punctuation usually adds noise in resume analysis.

In [12]:
import re

text_no_punct = re.sub(r"[^\w\s]", "", clean_text)

print(text_no_punct)


john doe
python developer
email john123gmailcom

skills python sql pandas numpy



#### 5.3 Tokenization

Tokenization breaks a sentence into individual words called tokens.

In [13]:
tokens = text_no_punct.split()

print(tokens)

['john', 'doe', 'python', 'developer', 'email', 'john123gmailcom', 'skills', 'python', 'sql', 'pandas', 'numpy']


#### 5.4 Stopword Removal

Stopwords are common words that often carry limited information
for a particular NLP task.

Examples:
- the
- is
- are
- and
- of
- in
- to

We will test stopword removal rather than automatically using it
in the final resume-screening pipeline.

In [14]:
import nltk

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Personal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

filtered_tokens = [
    word for word in tokens
    if word not in stop_words
]

print("Original tokens:")
print(tokens)

print("\nAfter stopword removal:")
print(filtered_tokens)

Original tokens:
['john', 'doe', 'python', 'developer', 'email', 'john123gmailcom', 'skills', 'python', 'sql', 'pandas', 'numpy']

After stopword removal:
['john', 'doe', 'python', 'developer', 'email', 'john123gmailcom', 'skills', 'python', 'sql', 'pandas', 'numpy']


#### 5.5 Stemming

Stemming reduces words to a common root by removing word endings.

It is fast, but the resulting root may not always be a
valid English word.

In [16]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

stemmed_tokens = [
    stemmer.stem(word)
    for word in filtered_tokens
]

print(stemmed_tokens)

['john', 'doe', 'python', 'develop', 'email', 'john123gmailcom', 'skill', 'python', 'sql', 'panda', 'numpi']


#### 5.6 Lemmatization

Lemmatization attempts to convert words to their meaningful
base form using linguistic information.

In [17]:
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Personal\AppData\Roaming\nltk_data...


[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Personal\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [18]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

lemmatized_tokens = [
    lemmatizer.lemmatize(word)
    for word in filtered_tokens
]
print("original token")
print(tokens)
print("\nAfter lemmatization:")
print(lemmatized_tokens)

original token
['john', 'doe', 'python', 'developer', 'email', 'john123gmailcom', 'skills', 'python', 'sql', 'pandas', 'numpy']

After lemmatization:
['john', 'doe', 'python', 'developer', 'email', 'john123gmailcom', 'skill', 'python', 'sql', 'panda', 'numpy']


In [19]:
# check lemmatization of technical terms 
technical_terms = [
    "Python",
    "Pandas",
    "NumPy",
    "developers",
    "machines",
    "learning",
    "models"
]

for word in technical_terms:
    print(f"{word} → {lemmatizer.lemmatize(word.lower())}")

Python → python
Pandas → panda
NumPy → numpy
developers → developer
machines → machine
learning → learning
models → model


### 6. Reusable Text Preprocessing Function

The preprocessing logic is moved into `src/preprocessing.py`
so that the same function can be reused throughout the project.

In [20]:
sample_resume = """
John Doe
Python Developer

Email: john.doe@gmail.com
GitHub: https://github.com/johndoe

Skills:
Python, C++, C#, SQL, Pandas, NumPy, Scikit-learn,
Node.js!!!
"""

cleaned_resume = preprocess_text(sample_resume)

print(cleaned_resume)

john doe python developer email github skills python c++ c# sql pandas numpy scikit-learn node.js


##### check sample pdf

In [21]:
from src.resume_parser import extract_resume_text
resume_path = "../data/raw/resumes/deepratim_ghosh_resume_2026_aug.pdf"

raw_resume_text = extract_resume_text(resume_path)

print(raw_resume_text)

DEEPRATIM GHOSH
Kolkata, India | 9163800261 | ghoshdeepratim752006@gmail.com
linkedin.com/in/deepratim-ghosh | github.com/dPro-123
PROFESSIONAL SUMMARY
Computer Science & Engineering undergraduate with an 8.92 CGPA and hands-on experience in Machine Learning, Data Analytics,
Python, SQL, and practical ML workflows. Built end-to-end ML projects involving EDA, feature engineering, model evaluation,
hyperparameter tuning, model interpretability, and deployment. Currently pursuing a training-cum-industrial internship in AI/ML and Data
Analytics, seeking internship opportunities in Machine Learning, Data Science, AI, or Data Analytics.
EDUCATION
Bachelor of Engineering (B.E.) — Computer Science & Engineering | Narula Institute of Technology    Expected 2028 | CGPA: 8.92
EXPERIENCE
Euphoria GenX — Training-cum-Industrial Intern (AI/ML & Data Analytics) | Hybrid | Aug 2026 – Present
• Working on practical applications of AI/ML and data analytics.
• Applying Python, data preprocessing, explora

### 7. Resume Text Preprocessing

The extracted resume text will now be passed through our
reusable preprocessing function before further analysis.

In [22]:
from src.preprocessing import preprocess_text

clean_resume_text = preprocess_text(raw_resume_text)

print(clean_resume_text)

deepratim ghosh kolkata india 9163800261 linkedin.com in deepratim-ghosh github.com dpro-123 professional summary computer science engineering undergraduate with an 8.92 cgpa and hands-on experience in machine learning data analytics python sql and practical ml workflows. built end-to-end ml projects involving eda feature engineering model evaluation hyperparameter tuning model interpretability and deployment. currently pursuing a training-cum-industrial internship in ai ml and data analytics seeking internship opportunities in machine learning data science ai or data analytics. education bachelor of engineering b.e. computer science engineering narula institute of technology expected 2028 cgpa 8.92 experience euphoria genx training-cum-industrial intern ai ml data analytics hybrid aug 2026 present working on practical applications of ai ml and data analytics. applying python data preprocessing exploratory analysis and machine learning concepts in an industry-oriented training environm

#### skill extraction using skill_extraction.py

In [23]:
from src.skill_extraction import extract_skills

resume_skills = extract_skills(clean_resume_text)

print("Detected Skills:")
print(resume_skills)

Detected Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql']


In [24]:
import importlib
import src.skill_extraction as se

importlib.reload(se)

<module 'src.skill_extraction' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\skill_extraction.py'>

In [25]:
extract_skills = se.extract_skills

In [26]:
## test with artificial test resume text
test_resume = """
Experienced in Python, ML, DL, sklearn, ReactJS,
NodeJS and Power BI.
"""

test_skills = extract_skills(test_resume)

print("Detected Skills:")
print(test_skills)

Detected Skills:
['deep learning', 'machine learning', 'node.js', 'power bi', 'python', 'react', 'scikit-learn']


In [27]:
## test with our sampel resume
resume_skills = extract_skills(clean_resume_text)

print("Detected Skills:")
print(resume_skills)

Detected Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql']


In [28]:
import importlib
import src.skill_extraction as se
importlib.reload(se)
extract_skills=se.extract_skills


In [29]:
resume_skills = extract_skills(clean_resume_text)

print("Detected Skills:")
print(resume_skills)

Detected Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql']


### test job description in the notebook

In [30]:
from src.job_parser import extract_job_description

job_path = "../data/raw/job_descriptions/data_science/data_science_01.txt"

raw_job_text = extract_job_description(job_path)

print(raw_job_text)

Data Scientist

We are looking for a Data Scientist to analyze complex datasets and
build predictive models. The role requires Python, Pandas, NumPy,
Scikit-learn, SQL, statistics, data visualization, and machine
learning. The candidate will perform exploratory data analysis,
feature engineering, model evaluation, and communicate insights to
stakeholders.


In [31]:
## preprocess job desription
clean_job_text = preprocess_text(raw_job_text)

print(clean_job_text)

data scientist we are looking for a data scientist to analyze complex datasets and build predictive models. the role requires python pandas numpy scikit-learn sql statistics data visualization and machine learning. the candidate will perform exploratory data analysis feature engineering model evaluation and communicate insights to stakeholders.


In [32]:
## extract skills from job description
job_skills = extract_skills(clean_job_text)

print("Job Skills:")
print(job_skills)

Job Skills:
['data analysis', 'machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql', 'statistics']


### 9. resume-job description matching

In [33]:
from src.skill_matching import compare_skills

In [34]:
skill_comparison = compare_skills(
    resume_skills,
    job_skills
)

print("Matched Skills:")
print(skill_comparison["matched_skills"])

print("\nMissing Skills:")
print(skill_comparison["missing_skills"])

print("\nExtra Skills:")
print(skill_comparison["extra_skills"])

Matched Skills:
['machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql']

Missing Skills:
['data analysis', 'statistics']

Extra Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript']


#### skill matching calculation

In [35]:
import importlib
import src.skill_matching as sk

importlib.reload(sk)

<module 'src.skill_matching' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\skill_matching.py'>

In [36]:
from src.skill_matching import calculate_skill_match_percentage

In [38]:
skill_match_score = calculate_skill_match_percentage(
    skill_comparison["matched_skills"],
    job_skills
)

print(f"Skill Match Score: {skill_match_score}%")

Skill Match Score: 75.0%


### 9.semantic similarity

In [39]:
import importlib
import src.semantic_matching as sm
importlib.reload(sm)

<module 'src.semantic_matching' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\semantic_matching.py'>

In [40]:
from src.semantic_matching import calculate_semantic_similarity

In [41]:
semantic_score = calculate_semantic_similarity(
    clean_resume_text,
    clean_job_text
)

print(f"Semantic Similarity Score: {semantic_score}%")

Semantic Similarity Score: 39.98%


### 10. Hybrid rsume - job match score
#### skill match score + semantic match score

In [42]:
import importlib
import match_score as ms
importlib.reload(ms)

ModuleNotFoundError: No module named 'match_score'

In [43]:
from src.match_score import calculate_hybrid_score

In [44]:
hybrid_score = calculate_hybrid_score(
    skill_match_score,
    semantic_score
)

print(f"Skill Match Score: {skill_match_score}%")
print(f"Semantic Similarity: {semantic_score}%")
print(f"Final Hybrid Match Score: {hybrid_score}%")

Skill Match Score: 75.0%
Semantic Similarity: 39.98%
Final Hybrid Match Score: 60.99%


### 11. skill-gap analyzer

In [45]:
import importlib
import src.skill_gap as skg
importlib.reload(skg)

<module 'src.skill_gap' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\skill_gap.py'>

In [46]:
from src.skill_gap import analyze_skill_gap

In [47]:
skill_gap_result = analyze_skill_gap(
    resume_skills,
    job_skills
)

print("Matched Skills:")
print(skill_gap_result["matched_skills"])

print("\nMissing Skills:")
print(skill_gap_result["missing_skills"])

print("\nExtra Skills:")
print(skill_gap_result["extra_skills"])

print(
    f"\nSkill Gap: "
    f"{skill_gap_result['skill_gap_percentage']}%"
)

Matched Skills:
['machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql']

Missing Skills:
['data analysis', 'statistics']

Extra Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript']

Skill Gap: 25.0%


### 12.skill-Gap recommendation

In [125]:
import importlib
import src.skill_recommendation as sr
importlib.reload(sr)

<module 'src.skill_recommendation' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\skill_recommendation.py'>

In [126]:
from src.skill_recommendation import generate_skill_recommendations

In [127]:
recommendations = generate_skill_recommendations(
    skill_gap_result["missing_skills"]
)

for recommendation in recommendations:
    print(
        f"Skill: {recommendation['skill']}"
    )
    print(
        f"Priority: {recommendation['priority']}"
    )
    print(
        f"Reason: {recommendation['reason']}"
    )
    print()

Skill: data analysis
Priority: High
Reason: Required by the target job

Skill: statistics
Priority: Medium
Reason: Required by the target job



### 13. test the loader

In [45]:
from src.job_loader import (
    load_job_dataset,
    load_job_description
)

In [46]:
## job dataset from job.csv
jobs_df = load_job_dataset(
    "../data/processed/jobs.csv"
)
print(jobs_df.loc[0,'file_path'])
print(jobs_df)

../data/raw/job_descriptions/ai_ml/ml_engineering_01.txt
   job_id                            job_title           domain  \
0   ML001              Machine Learning Intern            AI/ML   
1   ML002            Machine Learning Engineer            AI/ML   
2   ML003               Deep Learning Engineer            AI/ML   
3   ML004             Computer Vision Engineer            AI/ML   
4   ML005                          AI Engineer            AI/ML   
5   DS001                       Data Scientist     Data Science   
6   DS002                Junior Data Scientist     Data Science   
7   DS003                  Data Science Intern     Data Science   
8   DS004               Applied Data Scientist     Data Science   
9   DS005       Predictive Analytics Scientist     Data Science   
10  DA001                         Data Analyst   Data Analytics   
11  DA002                Business Data Analyst   Data Analytics   
12  DA003                Data Analytics Intern   Data Analytics   
13  D

In [47]:
## job description from the first job in the dataset
job_text = load_job_description(
    jobs_df.loc[0, "file_path"]
)

print(job_text)


Machine Learning Intern

We are looking for a Machine Learning Intern to work on
data-driven machine learning projects.

Responsibilities:
- Prepare and analyze datasets.
- Build and evaluate machine learning models.
- Perform data preprocessing and feature engineering.
- Work with Python and Scikit-learn.
- Query and manipulate data using SQL.

Requirements:
- Strong Python programming skills.
- Understanding of machine learning algorithms.
- Knowledge of SQL.
- Experience with Scikit-learn.
- Good analytical and problem-solving skills.


In [48]:
import importlib
import src.job_loader as jl

importlib.reload(jl)


<module 'src.job_loader' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\job_loader.py'>

In [49]:
from src.job_loader import load_all_jobs

In [50]:
jobs = load_all_jobs(
    "../data/processed/jobs.csv"
)

In [51]:
print("Number of jobs:", len(jobs))

Number of jobs: 25


In [52]:
print(jobs[0])

{'job_id': 'ML001', 'job_title': 'Machine Learning Intern', 'domain': 'AI/ML', 'file_path': '../data/raw/job_descriptions/ai_ml/ml_engineering_01.txt', 'required_skills': ['Python, Machine Learning, SQL, Scikit-learn'], 'description': '\nMachine Learning Intern\n\nWe are looking for a Machine Learning Intern to work on\ndata-driven machine learning projects.\n\nResponsibilities:\n- Prepare and analyze datasets.\n- Build and evaluate machine learning models.\n- Perform data preprocessing and feature engineering.\n- Work with Python and Scikit-learn.\n- Query and manipulate data using SQL.\n\nRequirements:\n- Strong Python programming skills.\n- Understanding of machine learning algorithms.\n- Knowledge of SQL.\n- Experience with Scikit-learn.\n- Good analytical and problem-solving skills.'}


In [53]:
from src.job_processing import process_job

In [54]:
processed_job = process_job(jobs[0])

print("Job Title:")
print(processed_job["job_title"])

print("\nExtracted Skills:")
print(processed_job["extracted_skills"])

print("\nRequired Skills:")
print(processed_job["required_skills"])

Job Title:
Machine Learning Intern

Extracted Skills:
['machine learning', 'python', 'scikit-learn', 'sql']

Required Skills:
['Python, Machine Learning, SQL, Scikit-learn']


##

#### process all jobs

In [55]:
import importlib
import src.job_processing as jp

importlib.reload(jp)

<module 'src.job_processing' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\job_processing.py'>

In [56]:
processed_jobs = jp.process_all_jobs(jobs)

print("Total processed jobs:", len(processed_jobs))

Total processed jobs: 25


In [57]:
print(processed_jobs[0]["job_title"])
print(processed_jobs[0]["extracted_skills"])

Machine Learning Intern
['machine learning', 'python', 'scikit-learn', 'sql']


In [58]:
import src.skill_matching as sm
import src.semantic_matching as sem
import src.match_score as ms

print(dir(sm))
print()
print(dir(sem))
print()
print(dir(ms))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_skill_match_percentage', 'compare_skills']

['TfidfVectorizer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_semantic_similarity', 'cosine_similarity']

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_hybrid_score']


## Step 26 — Multi-Job Matching Engine

### Objective

In this step, we will connect the existing resume analysis components
with our job dataset.

The goal is to calculate how well a resume matches **each job** in our
job database.

## Existing Components

We already have three important matching components:

1. **Skill Matching**
   - Compares resume skills with job skills.
   - Identifies matched, missing, and extra skills.
   - Calculates the skill matching percentage.

2. **Semantic Similarity**
   - Compares the meaning/content of the resume and job description.
   - Uses TF-IDF and Cosine Similarity.
   - Produces a semantic similarity score.

3. **Hybrid Score**
   - Combines the skill matching score and semantic similarity score.
   - Produces a final overall resume-job matching score.

## Current Functions

### Skill Matching
- `compare_skills()`
- `calculate_skill_match_percentage()`

### Semantic Similarity
- `calculate_semantic_similarity()`

### Hybrid Scoring
- `calculate_hybrid_score()`

## Target Pipeline

Resume
   ↓
Resume Skills + Resume Text
   ↓
Compare with every Job
   ↓
┌──────────────────────────────┐
│ Skill Matching               │
│ Semantic Similarity          │
│ Hybrid Score                 │
└──────────────────────────────┘
   ↓
Score for each job
   ↓
Rank jobs by overall score
   ↓
Top Job Recommendations

## Important

We will reuse the existing functions instead of rewriting the
matching logic.

This keeps the project modular, maintainable, and easier to debug.

In [59]:
print("Resume Skills:")
print(resume_skills)

print("\nNumber of Resume Skills:")
print(len(resume_skills))

print("\nClean Resume Text Length:")
print(len(clean_resume_text))

print("\nFirst 300 characters of Clean Resume Text:")
print(clean_resume_text[:300])

Resume Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql']

Number of Resume Skills:
14

Clean Resume Text Length:
2531

First 300 characters of Clean Resume Text:
deepratim ghosh kolkata india 9163800261 linkedin.com in deepratim-ghosh github.com dpro-123 professional summary computer science engineering undergraduate with an 8.92 cgpa and hands-on experience in machine learning data analytics python sql and practical ml workflows. built end-to-end ml project


In [60]:
variables = [
    name for name in globals()
    if not name.startswith("_")
]

print(variables)

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'np', 'pd', 'CountVectorizer', 'TfidfVectorizer', 'cosine_similarity', 'sys', 'os', 'preprocess_text', 'extract_resume_text', 'text', 'documents', 'vectorizer', 'bow_matrix', 'tfidf_vectorizer', 'tfidf_matrix', 'resume', 'job_description', 'similarity', 'score', 'raw_text', 'clean_text', 're', 'text_no_punct', 'tokens', 'nltk', 'stopwords', 'stop_words', 'filtered_tokens', 'PorterStemmer', 'stemmer', 'stemmed_tokens', 'WordNetLemmatizer', 'lemmatizer', 'lemmatized_tokens', 'technical_terms', 'word', 'sample_resume', 'cleaned_resume', 'resume_path', 'raw_resume_text', 'clean_resume_text', 'extract_skills', 'resume_skills', 'importlib', 'se', 'test_resume', 'test_skills', 'extract_job_description', 'job_path', 'raw_job_text', 'clean_job_text', 'job_skills', 'compare_skills', 'skill_comparison', 'sk', 'calculate_skill_match_percentage', 'skill_match_score', 'sm', 'calculate_semantic_similarity', 'semantic_score', 'calculate_hybrid_score

### 26.5 — Calculate Resume–Job Matching Scores

We will now compare the resume against the first processed job.

Three scores will be calculated:

1. Skill Matching Score
   - Compares resume skills with job skills.

2. Semantic Similarity Score
   - Compares the cleaned resume text with the cleaned job description
     using TF-IDF and cosine similarity.

3. Hybrid Score
   - Combines the skill matching score and semantic similarity score
     using the weights defined earlier.

This is the core scoring pipeline that will later be applied to every
job in the multi-domain job database.

In [61]:
skill_comparison = sm.compare_skills(
    resume_skills,
    processed_job["extracted_skills"]
)

In [62]:
matched_skills = skill_comparison["matched_skills"]
missing_skills = skill_comparison["missing_skills"]
extra_skills = skill_comparison["extra_skills"]

In [63]:
skill_match_score = sm.calculate_skill_match_percentage(
    matched_skills,
    processed_job["extracted_skills"]
)

In [64]:
print("Matched Skills:")
print(matched_skills)

print("\nMissing Skills:")
print(missing_skills)

print("\nExtra Skills:")
print(extra_skills)

print("\nSkill Match Score:")
print(f"{skill_match_score:.2f}%")

Matched Skills:
['machine learning', 'python', 'scikit-learn', 'sql']

Missing Skills:
[]

Extra Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']

Skill Match Score:
100.00%


In [65]:
import inspect

print(inspect.signature(sm.calculate_skill_match_percentage))

(matched_skills, job_skills)


In [66]:
import importlib
import src.skill_matching as sm

importlib.reload(sm)

<module 'src.skill_matching' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\skill_matching.py'>

###  26.5B — Calculate Semantic Similarity

Skill matching checks explicit skill overlap between the resume and
the job description.

Semantic similarity looks at the textual similarity between:

- `clean_resume_text` → processed resume
- `processed_job["clean_description"]` → processed job description

Our existing `calculate_semantic_similarity()` function uses TF-IDF
and Cosine Similarity.

The result represents how similar the resume and job description are
in terms of their textual content.

In [67]:
semantic_score = sem.calculate_semantic_similarity(
    clean_resume_text,
    processed_job["clean_description"]
)

print(f"Semantic Similarity: {semantic_score:.2f}%")

Semantic Similarity: 48.05%


### 26.5C — Calculate Hybrid Match Score

The hybrid score combines two complementary signals:

1. Skill Match Score
   - Measures explicit skill compatibility.

2. Semantic Similarity Score
   - Measures similarity between the resume content and job description.

We previously decided to give:

- Skill Weight = 0.60
- Semantic Weight = 0.40

Therefore:

Hybrid Score =
    (Skill Match × 0.60)
    +
    (Semantic Similarity × 0.40)

The hybrid score will eventually be used to rank jobs.

In [68]:
hybrid_score = ms.calculate_hybrid_score(
    skill_match_score,
    semantic_score
)

print(f"Skill Match Score: {skill_match_score:.2f}%")
print(f"Semantic Similarity: {semantic_score:.2f}%")
print(f"Hybrid Score: {hybrid_score:.2f}%")

Skill Match Score: 100.00%
Semantic Similarity: 48.05%
Hybrid Score: 79.22%


### 26.6 — Build a Reusable Job Scoring Function

The previous step calculated the matching scores manually for one job.

Now we will combine the complete scoring pipeline into one reusable
function.

The function will:

1. Compare resume skills with job skills.
2. Calculate the skill matching percentage.
3. Calculate semantic similarity between resume and job text.
4. Calculate the hybrid score.
5. Return all relevant results in a structured dictionary.

This function will later be called for every job in the database.

In [69]:
from src.job_matcher import score_job

In [70]:
job_result = score_job(
    resume_skills,
    clean_resume_text,
    processed_job
)

In [71]:
print("Job:", job_result["job_title"])
print("Domain:", job_result["domain"])

print("\nMatched Skills:")
print(job_result["matched_skills"])

print("\nMissing Skills:")
print(job_result["missing_skills"])

print("\nExtra Skills:")
print(job_result["extra_skills"])

print("\nSkill Match:", f"{job_result['skill_match_score']:.2f}%")
print("Semantic:", f"{job_result['semantic_score']:.2f}%")
print("Hybrid:", f"{job_result['hybrid_score']:.2f}%")

Job: Machine Learning Intern
Domain: AI/ML

Matched Skills:
['machine learning', 'python', 'scikit-learn', 'sql']

Missing Skills:
[]

Extra Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']

Skill Match: 100.00%
Semantic: 48.05%
Hybrid: 79.22%


### 26.7 — Score All Jobs

The `score_job()` function can evaluate one resume against one
processed job.

Our recommendation system must evaluate the same resume against
every job in the job database.

We will therefore:

1. Iterate through all processed jobs.
2. Calculate the complete matching score for each job.
3. Store every result in a list.
4. Sort the results by hybrid score.

This transforms our system from a single-job matcher into a
multi-job recommendation engine.

Pipeline:

Resume
   ↓
Processed Jobs
   ↓
Score each job
   ↓
Store results
   ↓
Rank by Hybrid Score
   ↓
Job Recommendations

In [72]:
all_job_results = []

for job in processed_jobs:
    result = score_job(
        resume_skills,
        clean_resume_text,
        job
    )

    all_job_results.append(result)

print("Total jobs scored:", len(all_job_results))

Total jobs scored: 25


In [73]:
print(all_job_results[0])

{'job_id': 'ML001', 'job_title': 'Machine Learning Intern', 'domain': 'AI/ML', 'matched_skills': ['machine learning', 'python', 'scikit-learn', 'sql'], 'missing_skills': [], 'extra_skills': ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas'], 'skill_match_score': 100.0, 'semantic_score': np.float64(48.05), 'hybrid_score': np.float64(79.22)}


### 26.7C — Rank Jobs by Hybrid Score

The hybrid score represents the overall compatibility between the
resume and a job.

We will sort all jobs in descending order so that the highest-scoring
jobs appear first.

The highest-scoring job becomes the strongest recommendation.

In [74]:
ranked_jobs = sorted(
    all_job_results,
    key=lambda x: x["hybrid_score"],
    reverse=True
)

In [75]:
## display top jobs
for i, job in enumerate(ranked_jobs[:10], start=1):

    print(
        f"{i}. {job['job_title']} | "
        f"{job['domain']} | "
        f"{job['hybrid_score']:.2f}%"
    )

1. Machine Learning Intern | AI/ML | 79.22%
2. Data Analyst | Data Analytics | 71.06%
3. Product Data Analyst | Data Analytics | 70.61%
4. Applied Data Scientist | Data Science | 66.92%
5. Machine Learning Engineer | AI/ML | 65.06%
6. AI Engineer | AI/ML | 64.50%
7. Predictive Analytics Scientist | Data Science | 62.90%
8. Data Science Intern | Data Science | 61.14%
9. Data Scientist | Data Science | 60.99%
10. Junior Data Scientist | Data Science | 60.86%


# Step 27 — Multi-Domain Job Dataset

## Objective

Our recommendation system should not be restricted to a single domain.

Instead, we will build a controlled multi-domain job dataset that
allows the system to recommend jobs from several related technology
domains.

## Selected Domains

We will initially cover:

1. AI / Machine Learning
2. Data Science
3. Data Analytics
4. NLP
5. Software Development

## Why Controlled Multi-Domain?

We do not want to add random job descriptions from hundreds of
different domains.

A controlled set of related domains allows us to:

- Keep the dataset manageable.
- Test the recommendation system properly.
- Compare recommendations across domains.
- Analyze skill gaps for different career paths.
- Demonstrate domain-aware job recommendations.

## Job Record Structure

Each job will contain:

- Job ID
- Job Title
- Domain
- Required Skills
- Job Description
- File Path

## Target Pipeline

Job Description Files
        ↓
Job Loader
        ↓
Preprocessing
        ↓
Skill Extraction
        ↓
Processed Job Dataset
        ↓
Resume–Job Matching
        ↓
Hybrid Score
        ↓
Ranked Recommendations

## 27.3 — Job Dataset Schema

Each job will follow a consistent structure.

| Field | Description |
|---|---|
| job_id | Unique identifier |
| job_title | Name of the job |
| domain | Career domain |
| required_skills | Skills expected for the role |
| description | Full job description |
| file_path | Location of the original JD file |

Example:

job_id:
ML001

job_title:
Machine Learning Engineer

domain:
AI/ML

required_skills:
Python, Machine Learning, SQL, Scikit-learn

description:
A Machine Learning Engineer is responsible for developing,
training, evaluating, and deploying machine learning models...

## 27.5 — Create AI/ML Job Descriptions

We will now expand the raw job-description dataset.

Each job will be stored as an individual `.txt` file inside the
appropriate domain folder.

The job metadata will be stored separately in `jobs.csv`.

The raw job-description files are the source of the actual job
content, while `jobs.csv` acts as the structured index of available
jobs.

Structure:

raw job description (.txt)
        ↓
jobs.csv metadata
        ↓
job loader
        ↓
job processing
        ↓
recommendation engine

In [76]:
print("Columns:")
print(jobs_df.columns.tolist())

print("\nNumber of jobs:")
print(len(jobs_df))

print("\nCurrent dataset:")
display(jobs_df)

Columns:
['job_id', 'job_title', 'domain', 'file_path', 'required_skills']

Number of jobs:
25

Current dataset:


,job_id,job_title,domain,file_path,required_skills
0,ML001,Machine Learning Intern,AI/ML,../data/raw/job_descriptions/ai_ml/ml_engineer...,"Python, Machine Learning, SQL, Scikit-learn"
1,ML002,Machine Learning Engineer,AI/ML,../data/raw/job_descriptions/ai_ml/ml_engineer...,"Python, Machine Learning, Scikit-learn, Tensor..."
2,ML003,Deep Learning Engineer,AI/ML,../data/raw/job_descriptions/ai_ml/ml_engineer...,"Python, Deep Learning, TensorFlow, PyTorch, Git"
3,ML004,Computer Vision Engineer,AI/ML,../data/raw/job_descriptions/ai_ml/ml_engineer...,"Python, Computer Vision, OpenCV, Deep Learning..."
4,ML005,AI Engineer,AI/ML,../data/raw/job_descriptions/ai_ml/ml_engineer...,"Python, Machine Learning, Deep Learning, Tenso..."
5,DS001,Data Scientist,Data Science,../data/raw/job_descriptions/data_science/data...,"Python, Pandas, NumPy, Scikit-learn, SQL, Stat..."
6,DS002,Junior Data Scientist,Data Science,../data/raw/job_descriptions/data_science/data...,"Python, Pandas, NumPy, SQL, Statistics, Scikit..."
7,DS003,Data Science Intern,Data Science,../data/raw/job_descriptions/data_science/data...,"Python, Pandas, NumPy, SQL, Statistics, Scikit..."
8,DS004,Applied Data Scientist,Data Science,../data/raw/job_descriptions/data_science/data...,"Python, Pandas, NumPy, Scikit-learn, SQL, Mach..."
9,DS005,Predictive Analytics Scientist,Data Science,../data/raw/job_descriptions/data_science/data...,"Python, SQL, Pandas, Statistics, Machine Learn..."


#### 27.5A — Generate AI/ML Job Description Files

The following cell creates the remaining AI/ML job-description files
inside the raw dataset directory.

The descriptions are synthetic but realistic and are designed to
provide controlled skill overlap for testing the recommendation
system.

In [80]:
from pathlib import Path

ai_ml_dir = Path("../data/raw/job_descriptions/ai_ml")

ai_ml_dir.mkdir(parents=True, exist_ok=True)

ai_ml_jobs = {
    "ml_engineering_02.txt": """
Machine Learning Engineer

We are looking for a Machine Learning Engineer to design, train,
evaluate, and improve machine learning models. The role involves
Python programming, data preprocessing, feature engineering,
machine learning algorithms, Scikit-learn, TensorFlow, model
evaluation, and Git-based version control. Experience with model
deployment and production machine learning workflows is desirable.
""",

    "ml_engineering_03.txt": """
Deep Learning Engineer

We are seeking a Deep Learning Engineer to develop neural-network
based solutions. The role involves Python programming, deep learning,
neural networks, model training, TensorFlow, PyTorch, computer vision,
experimentation, and Git. Candidates should understand model
evaluation and optimization techniques.
""",

    "ml_engineering_04.txt": """
Computer Vision Engineer

We are looking for a Computer Vision Engineer to develop image and
video processing applications. The role requires Python, computer
vision, OpenCV, deep learning, PyTorch, image classification, object
detection, and model evaluation. Experience with image preprocessing
and neural networks is preferred.
""",

    "ml_engineering_05.txt": """
AI Engineer

The AI Engineer will develop intelligent applications using machine
learning and deep learning techniques. The position requires Python,
machine learning, deep learning, TensorFlow, data preprocessing,
model evaluation, and Git. Experience with deploying AI models and
working with real-world datasets is desirable.
"""
}

for filename, description in ai_ml_jobs.items():
    file_path = ai_ml_dir / filename

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(description.strip())

print("AI/ML job files created successfully.")

for file_path in sorted(ai_ml_dir.glob("*.txt")):
    print(file_path.name)

AI/ML job files created successfully.
ml_engineering_01.txt
ml_engineering_02.txt
ml_engineering_03.txt
ml_engineering_04.txt
ml_engineering_05.txt


In [81]:
ai_ml_files = sorted(ai_ml_dir.glob("*.txt"))

print("Total AI/ML job files:", len(ai_ml_files))

for file_path in ai_ml_files:
    print("-", file_path.name)

Total AI/ML job files: 5
- ml_engineering_01.txt
- ml_engineering_02.txt
- ml_engineering_03.txt
- ml_engineering_04.txt
- ml_engineering_05.txt


#### 27.6 — Create Data Science Job Description Files

We will create five controlled Data Science job descriptions.

The jobs will have overlapping skills with AI/ML and Data Analytics,
but will emphasize data analysis, statistics, experimentation,
visualization, and predictive modeling.

This controlled overlap allows the recommendation system to
distinguish between related domains.

In [82]:
from pathlib import Path

data_science_dir = Path("../data/raw/job_descriptions/data_science")

data_science_dir.mkdir(parents=True, exist_ok=True)

data_science_jobs = {
    "data_science_01.txt": """
Data Scientist

We are looking for a Data Scientist to analyze complex datasets and
build predictive models. The role requires Python, Pandas, NumPy,
Scikit-learn, SQL, statistics, data visualization, and machine
learning. The candidate will perform exploratory data analysis,
feature engineering, model evaluation, and communicate insights to
stakeholders.
""",

    "data_science_02.txt": """
Junior Data Scientist

The Junior Data Scientist will support data analysis and predictive
modeling projects. Responsibilities include Python programming,
Pandas, NumPy, SQL, exploratory data analysis, statistics, data
visualization, and Scikit-learn. Knowledge of machine learning and
Git is desirable.
""",

    "data_science_03.txt": """
Data Science Intern

We are seeking a Data Science Intern to work with real-world
datasets. The role involves Python, Pandas, NumPy, SQL, data
cleaning, exploratory data analysis, visualization, statistics, and
basic machine learning. Experience with Jupyter Notebook and
Scikit-learn is preferred.
""",

    "data_science_04.txt": """
Applied Data Scientist

The Applied Data Scientist will develop data-driven solutions using
Python and machine learning. The position requires Pandas, NumPy,
Scikit-learn, SQL, statistics, feature engineering, predictive
modeling, data visualization, and model evaluation. Strong analytical
and problem-solving skills are expected.
""",

    "data_science_05.txt": """
Predictive Analytics Scientist

We are looking for a Predictive Analytics Scientist to build
predictive models and extract insights from structured datasets.
Required skills include Python, SQL, Pandas, statistics, machine
learning, Scikit-learn, feature engineering, and data visualization.
Experience with predictive analytics and model evaluation is
preferred.
"""
}

for filename, description in data_science_jobs.items():
    file_path = data_science_dir / filename

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(description.strip())

print("Data Science job files created successfully.")

Data Science job files created successfully.


In [83]:
data_science_files = sorted(data_science_dir.glob("*.txt"))

print("Total Data Science job files:", len(data_science_files))

for file_path in data_science_files:
    print("-", file_path.name)

Total Data Science job files: 5
- data_science_01.txt
- data_science_02.txt
- data_science_03.txt
- data_science_04.txt
- data_science_05.txt


#### 27.7 — Create Data Analytics Job Description Files

We will create five controlled Data Analytics job descriptions.

These roles will emphasize SQL, Excel, business intelligence,
dashboarding, data visualization, reporting, and business insights.

Some roles may also require Python and Pandas.

The overlap with Data Science is intentional because real-world
technology careers often share common skills.

In [84]:
from pathlib import Path

data_analytics_dir = Path("../data/raw/job_descriptions/data_analytics")

data_analytics_dir.mkdir(parents=True, exist_ok=True)

data_analytics_jobs = {
    "data_analytics_01.txt": """
Data Analyst

We are looking for a Data Analyst to analyze business data and
generate actionable insights. The role requires SQL, Excel, Python,
Pandas, data cleaning, data visualization, and reporting. The
candidate will create dashboards, analyze KPIs, identify trends,
and communicate findings to business stakeholders.
""",

    "data_analytics_02.txt": """
Business Data Analyst

The Business Data Analyst will analyze operational and business
datasets to support decision-making. Required skills include SQL,
Excel, Power BI, data visualization, reporting, KPI analysis, and
statistical analysis. Knowledge of Python and Pandas is desirable.
""",

    "data_analytics_03.txt": """
Data Analytics Intern

We are seeking a Data Analytics Intern to support data preparation,
analysis, and reporting tasks. The position involves SQL, Excel,
Python, Pandas, data visualization, dashboards, and exploratory data
analysis. The candidate will assist in identifying trends and
generating business insights.
""",

    "data_analytics_04.txt": """
Business Intelligence Analyst

The Business Intelligence Analyst will develop dashboards and
reports to support business decisions. The role requires SQL,
Power BI, Tableau, Excel, data visualization, KPI development, and
business analysis. Experience with data warehousing and reporting
is beneficial.
""",

    "data_analytics_05.txt": """
Product Data Analyst

The Product Data Analyst will analyze product usage and customer
behavior data. Responsibilities include SQL, Excel, Python, Pandas,
data visualization, KPI analysis, A/B testing, and reporting.
The candidate will work with product teams to identify trends and
provide data-driven recommendations.
"""
}

for filename, description in data_analytics_jobs.items():
    file_path = data_analytics_dir / filename

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(description.strip())

print("Data Analytics job files created successfully.")

Data Analytics job files created successfully.


In [85]:
data_analytics_files = sorted(data_analytics_dir.glob("*.txt"))

print("Total Data Analytics job files:", len(data_analytics_files))

for file_path in data_analytics_files:
    print("-", file_path.name)

Total Data Analytics job files: 5
- data_analytics_01.txt
- data_analytics_02.txt
- data_analytics_03.txt
- data_analytics_04.txt
- data_analytics_05.txt


#### 27.8 — Create Research Job Description Files

We will create five controlled Research job descriptions.

These roles will focus on research methodology, experimentation,
algorithm development, technical literature, scientific writing,
data analysis, and machine learning research.

Some research roles will overlap with AI/ML and Data Science because
those skills are commonly used in technical research.

In [86]:
from pathlib import Path

research_dir = Path("../data/raw/job_descriptions/research")

research_dir.mkdir(parents=True, exist_ok=True)

research_jobs = {
    "research_01.txt": """
AI Research Intern

We are looking for an AI Research Intern to assist with research
projects involving machine learning and artificial intelligence.
The role requires Python, machine learning, data analysis,
experimentation, literature review, and scientific documentation.
The candidate will implement algorithms, evaluate experiments, and
analyze research results.
""",

    "research_02.txt": """
Machine Learning Research Assistant

The Machine Learning Research Assistant will support experimental
research in machine learning. Responsibilities include Python,
machine learning, statistics, data analysis, algorithm
implementation, experimentation, and technical literature review.
Experience with Scikit-learn and scientific computing is desirable.
""",

    "research_03.txt": """
Computer Science Research Intern

We are seeking a Computer Science Research Intern to assist in
algorithm development and experimental evaluation. The role involves
Python, algorithms, data structures, research methodology,
experimentation, technical writing, and data analysis. The candidate
will review research papers and document experimental findings.
""",

    "research_04.txt": """
Data Science Researcher

The Data Science Researcher will investigate data-driven methods and
develop experimental models. Required skills include Python,
statistics, machine learning, Pandas, NumPy, data analysis,
experimental design, and scientific communication. Experience with
research papers and reproducible experiments is preferred.
""",

    "research_05.txt": """
AI Research Engineer

The AI Research Engineer will develop and evaluate novel machine
learning approaches through experimentation. The role requires
Python, machine learning, deep learning, PyTorch, algorithm
development, model evaluation, literature review, and technical
documentation. Strong analytical and problem-solving skills are
expected.
"""
}

for filename, description in research_jobs.items():
    file_path = research_dir / filename

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(description.strip())

print("Research job files created successfully.")

Research job files created successfully.


In [87]:
research_files = sorted(research_dir.glob("*.txt"))

print("Total Research job files:", len(research_files))

for file_path in research_files:
    print("-", file_path.name)

Total Research job files: 5
- research_01.txt
- research_02.txt
- research_03.txt
- research_04.txt
- research_05.txt


#### 27.9 — Create Web Development Job Description Files

We will create five controlled Web Development job descriptions.

The roles will cover frontend, backend, full-stack, and general web
development.

The descriptions will contain overlapping skills such as JavaScript,
React, Node.js, Python, REST APIs, SQL, Git, and databases.

This gives the recommendation system another distinct career path
while still allowing transferable skills to influence matching.

In [88]:
from pathlib import Path

web_dev_dir = Path("../data/raw/job_descriptions/web_development")

web_dev_dir.mkdir(parents=True, exist_ok=True)

web_development_jobs = {
    "web_development_01.txt": """
Frontend Developer

We are looking for a Frontend Developer to build responsive and
interactive web applications. The role requires HTML, CSS,
JavaScript, React, Git, and REST API integration. The candidate
should understand component-based development, responsive design,
and modern frontend development practices.
""",

    "web_development_02.txt": """
Backend Developer

The Backend Developer will design and develop server-side
applications and APIs. Required skills include Python, Node.js,
REST APIs, SQL, databases, Git, and backend development. The
candidate will build scalable services, implement APIs, and work
with relational databases.
""",

    "web_development_03.txt": """
Full Stack Developer

We are seeking a Full Stack Developer to develop complete web
applications. The role requires JavaScript, React, Node.js, Python,
SQL, REST APIs, Git, HTML, and CSS. The candidate will work across
frontend and backend components and integrate web applications with
databases.
""",

    "web_development_04.txt": """
Python Web Developer

The Python Web Developer will build and maintain web applications
using Python-based technologies. Required skills include Python,
Flask, REST APIs, SQL, HTML, CSS, JavaScript, and Git. Experience
with backend development, databases, and web application deployment
is desirable.
""",

    "web_development_05.txt": """
Web Development Intern

We are looking for a Web Development Intern to assist in building
and maintaining web applications. The role involves HTML, CSS,
JavaScript, React, Python, SQL, Git, and REST APIs. The candidate
will support frontend development, backend integration, debugging,
and basic database operations.
"""
}

for filename, description in web_development_jobs.items():
    file_path = web_dev_dir / filename

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(description.strip())

print("Web Development job files created successfully.")

Web Development job files created successfully.


In [89]:
web_dev_files = sorted(web_dev_dir.glob("*.txt"))

print("Total Web Development job files:", len(web_dev_files))

for file_path in web_dev_files:
    print("-", file_path.name)

Total Web Development job files: 5
- web_development_01.txt
- web_development_02.txt
- web_development_03.txt
- web_development_04.txt
- web_development_05.txt


#### 27.10 — Verify the Complete Raw Job Dataset

Before generating `jobs.csv`, we verify that all five domains contain
exactly five job descriptions.

Expected total:

5 domains × 5 jobs = 25 job descriptions.

In [90]:
domains = {
    "AI/ML": Path("../data/raw/job_descriptions/ai_ml"),
    "Data Science": Path("../data/raw/job_descriptions/data_science"),
    "Data Analytics": Path("../data/raw/job_descriptions/data_analytics"),
    "Research": Path("../data/raw/job_descriptions/research"),
    "Web Development": Path("../data/raw/job_descriptions/web_development")
}

total_files = 0

for domain, folder in domains.items():
    files = list(folder.glob("*.txt"))
    print(f"{domain}: {len(files)} files")
    total_files += len(files)

print("\nTotal job descriptions:", total_files)

AI/ML: 5 files
Data Science: 5 files
Data Analytics: 5 files
Research: 5 files
Web Development: 5 files

Total job descriptions: 25


### 27.11 — Automatically Generate `jobs.csv`

All 25 raw job-description files are now available.

Instead of manually entering job metadata, we will scan the domain
folders and generate `jobs.csv` programmatically.

The CSV will act as an index for the raw job-description files.

Each record will contain:

- job_id
- job_title
- domain
- file_path
- required_skills

The actual job description remains in its corresponding `.txt` file.

In [77]:
## define the job metadata dictionary
job_metadata = {
    "ai_ml": {
        "domain": "AI/ML",
        "jobs": {
            "ml_engineering_01.txt": {
                "job_id": "ML001",
                "job_title": "Machine Learning Intern",
                "required_skills": "Python, Machine Learning, SQL, Scikit-learn"
            },
            "ml_engineering_02.txt": {
                "job_id": "ML002",
                "job_title": "Machine Learning Engineer",
                "required_skills": "Python, Machine Learning, Scikit-learn, TensorFlow, Git"
            },
            "ml_engineering_03.txt": {
                "job_id": "ML003",
                "job_title": "Deep Learning Engineer",
                "required_skills": "Python, Deep Learning, TensorFlow, PyTorch, Git"
            },
            "ml_engineering_04.txt": {
                "job_id": "ML004",
                "job_title": "Computer Vision Engineer",
                "required_skills": "Python, Computer Vision, OpenCV, Deep Learning, PyTorch"
            },
            "ml_engineering_05.txt": {
                "job_id": "ML005",
                "job_title": "AI Engineer",
                "required_skills": "Python, Machine Learning, Deep Learning, TensorFlow, Git"
            }
        }
    },

    "data_science": {
        "domain": "Data Science",
        "jobs": {
            "data_science_01.txt": {
                "job_id": "DS001",
                "job_title": "Data Scientist",
                "required_skills": "Python, Pandas, NumPy, Scikit-learn, SQL, Statistics"
            },
            "data_science_02.txt": {
                "job_id": "DS002",
                "job_title": "Junior Data Scientist",
                "required_skills": "Python, Pandas, NumPy, SQL, Statistics, Scikit-learn"
            },
            "data_science_03.txt": {
                "job_id": "DS003",
                "job_title": "Data Science Intern",
                "required_skills": "Python, Pandas, NumPy, SQL, Statistics, Scikit-learn"
            },
            "data_science_04.txt": {
                "job_id": "DS004",
                "job_title": "Applied Data Scientist",
                "required_skills": "Python, Pandas, NumPy, Scikit-learn, SQL, Machine Learning"
            },
            "data_science_05.txt": {
                "job_id": "DS005",
                "job_title": "Predictive Analytics Scientist",
                "required_skills": "Python, SQL, Pandas, Statistics, Machine Learning, Scikit-learn"
            }
        }
    },

    "data_analytics": {
        "domain": "Data Analytics",
        "jobs": {
            "data_analytics_01.txt": {
                "job_id": "DA001",
                "job_title": "Data Analyst",
                "required_skills": "SQL, Excel, Python, Pandas, Data Visualization"
            },
            "data_analytics_02.txt": {
                "job_id": "DA002",
                "job_title": "Business Data Analyst",
                "required_skills": "SQL, Excel, Power BI, Data Visualization, Statistics"
            },
            "data_analytics_03.txt": {
                "job_id": "DA003",
                "job_title": "Data Analytics Intern",
                "required_skills": "SQL, Excel, Python, Pandas, Data Visualization"
            },
            "data_analytics_04.txt": {
                "job_id": "DA004",
                "job_title": "Business Intelligence Analyst",
                "required_skills": "SQL, Power BI, Tableau, Excel, Data Visualization"
            },
            "data_analytics_05.txt": {
                "job_id": "DA005",
                "job_title": "Product Data Analyst",
                "required_skills": "SQL, Excel, Python, Pandas, A/B Testing"
            }
        }
    },

    "research": {
        "domain": "Research",
        "jobs": {
            "research_01.txt": {
                "job_id": "RS001",
                "job_title": "AI Research Intern",
                "required_skills": "Python, Machine Learning, Data Analysis, Experimentation"
            },
            "research_02.txt": {
                "job_id": "RS002",
                "job_title": "Machine Learning Research Assistant",
                "required_skills": "Python, Machine Learning, Statistics, Data Analysis"
            },
            "research_03.txt": {
                "job_id": "RS003",
                "job_title": "Computer Science Research Intern",
                "required_skills": "Python, Algorithms, Data Structures, Research Methodology"
            },
            "research_04.txt": {
                "job_id": "RS004",
                "job_title": "Data Science Researcher",
                "required_skills": "Python, Statistics, Machine Learning, Pandas, NumPy"
            },
            "research_05.txt": {
                "job_id": "RS005",
                "job_title": "AI Research Engineer",
                "required_skills": "Python, Machine Learning, Deep Learning, PyTorch"
            }
        }
    },

    "web_development": {
        "domain": "Web Development",
        "jobs": {
            "web_development_01.txt": {
                "job_id": "WD001",
                "job_title": "Frontend Developer",
                "required_skills": "HTML, CSS, JavaScript, React, Git"
            },
            "web_development_02.txt": {
                "job_id": "WD002",
                "job_title": "Backend Developer",
                "required_skills": "Python, Node.js, REST APIs, SQL, Git"
            },
            "web_development_03.txt": {
                "job_id": "WD003",
                "job_title": "Full Stack Developer",
                "required_skills": "JavaScript, React, Node.js, Python, SQL, REST APIs"
            },
            "web_development_04.txt": {
                "job_id": "WD004",
                "job_title": "Python Web Developer",
                "required_skills": "Python, Flask, REST APIs, SQL, JavaScript"
            },
            "web_development_05.txt": {
                "job_id": "WD005",
                "job_title": "Web Development Intern",
                "required_skills": "HTML, CSS, JavaScript, React, Python, SQL"
            }
        }
    }
}

In [86]:
## generate the csv
import pandas as pd
from pathlib import Path

records = []

base_path = Path("../data/raw/job_descriptions")

for folder_name, domain_info in job_metadata.items():

    domain = domain_info["domain"]

    for filename, metadata in domain_info["jobs"].items():

        file_path = base_path / folder_name / filename

        records.append({
            "job_id": metadata["job_id"],
            "job_title": metadata["job_title"],
            "domain": domain,
            "file_path": str(file_path).replace("\\", "/"),
            "required_skills": metadata["required_skills"]
        })

jobs_df = pd.DataFrame(records)

jobs_csv_path = Path("../data/processed/jobs.csv")

jobs_df.to_csv(
    jobs_csv_path,
    index=False
)

print("jobs.csv generated successfully.")
print("Total jobs:", len(jobs_df))

jobs.csv generated successfully.
Total jobs: 25


In [87]:
print(jobs_df.shape)

print(jobs_df["domain"].value_counts())

(25, 5)
domain
AI/ML              5
Data Science       5
Data Analytics     5
Research           5
Web Development    5
Name: count, dtype: int64


In [88]:
csv_path = "../data/processed/jobs.csv"

jobs = load_all_jobs(csv_path)

print("Total jobs loaded:", len(jobs))

Total jobs loaded: 25


In [80]:
print (jobs[0])

{'job_id': 'ML001', 'job_title': 'Machine Learning Intern', 'domain': 'AI/ML', 'file_path': '../data/raw/job_descriptions/ai_ml/ml_engineering_01.txt', 'required_skills': ['Python, Machine Learning, SQL, Scikit-learn'], 'description': '\nMachine Learning Intern\n\nWe are looking for a Machine Learning Intern to work on\ndata-driven machine learning projects.\n\nResponsibilities:\n- Prepare and analyze datasets.\n- Build and evaluate machine learning models.\n- Perform data preprocessing and feature engineering.\n- Work with Python and Scikit-learn.\n- Query and manipulate data using SQL.\n\nRequirements:\n- Strong Python programming skills.\n- Understanding of machine learning algorithms.\n- Knowledge of SQL.\n- Experience with Scikit-learn.\n- Good analytical and problem-solving skills.'}


In [81]:
## processed all jobs
processed_jobs = []

for job in jobs:
    processed_job = process_job(job)
    processed_jobs.append(processed_job)

print("Total processed jobs:", len(processed_jobs))

Total processed jobs: 25


In [82]:
print (processed_jobs[0])

{'job_id': 'ML001', 'job_title': 'Machine Learning Intern', 'domain': 'AI/ML', 'file_path': '../data/raw/job_descriptions/ai_ml/ml_engineering_01.txt', 'required_skills': ['Python, Machine Learning, SQL, Scikit-learn'], 'description': '\nMachine Learning Intern\n\nWe are looking for a Machine Learning Intern to work on\ndata-driven machine learning projects.\n\nResponsibilities:\n- Prepare and analyze datasets.\n- Build and evaluate machine learning models.\n- Perform data preprocessing and feature engineering.\n- Work with Python and Scikit-learn.\n- Query and manipulate data using SQL.\n\nRequirements:\n- Strong Python programming skills.\n- Understanding of machine learning algorithms.\n- Knowledge of SQL.\n- Experience with Scikit-learn.\n- Good analytical and problem-solving skills.', 'clean_description': 'machine learning intern we are looking for a machine learning intern to work on data-driven machine learning projects. responsibilities - prepare and analyze datasets. - build a

In [83]:
print(processed_jobs[0]["job_title"])
print(processed_jobs[0]["extracted_skills"])

Machine Learning Intern
['machine learning', 'python', 'scikit-learn', 'sql']


### Step 27.14 — Score All Jobs

We will calculate a hybrid compatibility score between the resume
and every processed job.

For each job we calculate:

1. Skill Matching Score
2. Semantic Similarity Score
3. Hybrid Score

The resulting scores will be used to rank the jobs.

The highest hybrid score represents the strongest resume-job
compatibility according to our scoring methodology.

In [84]:
print(type(cleaned_resume))
print(len(cleaned_resume))

<class 'str'>
97


In [89]:
results = []

for job in processed_jobs:

    score = score_job(
        resume_skills,
        cleaned_resume,
        job
    )

    results.append({
        "job_id": job["job_id"],
        "job_title": job["job_title"],
        "domain": job["domain"],
        "skill_match_score": score["skill_match_score"],
        "semantic_score": score["semantic_score"],
        "hybrid_score": score["hybrid_score"]
    })

print("Jobs scored:", len(results))

Jobs scored: 25


In [90]:
import inspect
print(inspect.signature(score_job))

(resume_skills, resume_text, job)


In [91]:
results_df = pd.DataFrame(results)

print("Shape:", results_df.shape)

#display(results_df)

Shape: (25, 6)


In [92]:
### rank the jobs
ranked_jobs = results_df.sort_values(
    by="hybrid_score",
    ascending=False
).reset_index(drop=True)
print("shape",ranked_jobs.shape)
#display(ranked_jobs)

shape (25, 6)


In [93]:
top_jobs = ranked_jobs.head(10)

display(
    top_jobs[
        [
            "job_title",
            "domain",
            "skill_match_score",
            "semantic_score",
            "hybrid_score"
        ]
    ]
)


,job_title,domain,skill_match_score,semantic_score,hybrid_score
0,Machine Learning Intern,AI/ML,100.00,13.55,65.42
1,Data Analyst,Data Analytics,100.00,6.10,62.44
2,Product Data Analyst,Data Analytics,100.00,5.84,62.34
3,Applied Data Scientist,Data Science,85.71,14.63,57.28
4,Predictive Analytics Scientist,Data Science,83.33,12.12,54.85
5,Data Science Intern,Data Science,77.78,13.02,51.88
6,Junior Data Scientist,Data Science,77.78,12.12,51.52
7,Backend Developer,Web Development,75.00,13.80,50.52
8,Machine Learning Engineer,AI/ML,80.00,5.39,50.16
9,Data Scientist,Data Science,75.00,12.12,49.85


In [94]:
display("top_jobs shape:",top_jobs.shape)

'top_jobs shape:'

(10, 6)

### 27.15 — Domain-Level Recommendation Analysis

The recommendation system contains five career domains.

We will aggregate the hybrid scores of jobs belonging to each domain
to identify which career domains are the strongest matches for the
resume.

This provides a higher-level career recommendation in addition to
individual job recommendations.

In [95]:
domain_scores = (
    results_df
    .groupby("domain")["hybrid_score"]
    .mean()
    .sort_values(ascending=False)
)

print("Domain Recommendation Scores:")
display(domain_scores)

Domain Recommendation Scores:


domain
Data Science       53.076
Data Analytics     48.296
AI/ML              45.474
Research           40.856
Web Development    37.978
Name: hybrid_score, dtype: float64

In [96]:
display(
    results_df[
        [
            "job_title",
            "domain",
            "skill_match_score",
            "semantic_score",
            "hybrid_score"
        ]
    ].sort_values(
        by="hybrid_score",
        ascending=False
    )
)

,job_title,domain,skill_match_score,semantic_score,hybrid_score
0,Machine Learning Intern,AI/ML,100.00,13.55,65.42
10,Data Analyst,Data Analytics,100.00,6.10,62.44
14,Product Data Analyst,Data Analytics,100.00,5.84,62.34
8,Applied Data Scientist,Data Science,85.71,14.63,57.28
9,Predictive Analytics Scientist,Data Science,83.33,12.12,54.85
7,Data Science Intern,Data Science,77.78,13.02,51.88
6,Junior Data Scientist,Data Science,77.78,12.12,51.52
21,Backend Developer,Web Development,75.00,13.80,50.52
1,Machine Learning Engineer,AI/ML,80.00,5.39,50.16
5,Data Scientist,Data Science,75.00,12.12,49.85


In [97]:
test_job = processed_jobs[0]

print("Job:", test_job["job_title"])
print()
print("Resume length:", len(cleaned_resume))
print("Job description length:", len(test_job["description"]))

Job: Machine Learning Intern

Resume length: 97
Job description length: 544


In [98]:
test_semantic = calculate_semantic_similarity(
    cleaned_resume,
    test_job["description"]
)

print("Independent Semantic Similarity:", test_semantic)

Independent Semantic Similarity: 13.55


In [99]:
best_domain = domain_scores.index[0]
best_domain_score = domain_scores.iloc[0]

print("Recommended Career Domain:", best_domain)
print("Average Compatibility Score:", round(best_domain_score, 2), "%")

Recommended Career Domain: Data Science
Average Compatibility Score: 53.08 %


### 27.17 — Generate Top Job Recommendations

The system has scored and ranked all 25 jobs.

We will now generate a concise recommendation table containing the
highest-ranked jobs and their compatibility scores.

The recommendation is based on the hybrid score, which combines
skill matching and semantic similarity.

In [100]:
recommendation_table = ranked_jobs[
    [
        "job_title",
        "domain",
        "skill_match_score",
        "semantic_score",
        "hybrid_score"
    ]
].head(10).copy()

recommendation_table.columns = [
    "Job Title",
    "Domain",
    "Skill Match (%)",
    "Semantic Similarity (%)",
    "Hybrid Score (%)"
]

display(recommendation_table)

,Job Title,Domain,Skill Match (%),Semantic Similarity (%),Hybrid Score (%)
0,Machine Learning Intern,AI/ML,100.00,13.55,65.42
1,Data Analyst,Data Analytics,100.00,6.10,62.44
2,Product Data Analyst,Data Analytics,100.00,5.84,62.34
3,Applied Data Scientist,Data Science,85.71,14.63,57.28
4,Predictive Analytics Scientist,Data Science,83.33,12.12,54.85
5,Data Science Intern,Data Science,77.78,13.02,51.88
6,Junior Data Scientist,Data Science,77.78,12.12,51.52
7,Backend Developer,Web Development,75.00,13.80,50.52
8,Machine Learning Engineer,AI/ML,80.00,5.39,50.16
9,Data Scientist,Data Science,75.00,12.12,49.85


### 27.17 — Generate Top Job Recommendations

The system has scored and ranked all 25 jobs.

We will now generate a concise recommendation table containing the
highest-ranked jobs and their compatibility scores.

The recommendation is based on the hybrid score, which combines
skill matching and semantic similarity.

In [101]:
recommendation_table = ranked_jobs[
    [
        "job_title",
        "domain",
        "skill_match_score",
        "semantic_score",
        "hybrid_score"
    ]
].head(10).copy()

recommendation_table.columns = [
    "Job Title",
    "Domain",
    "Skill Match (%)",
    "Semantic Similarity (%)",
    "Hybrid Score (%)"
]

display(recommendation_table)

,Job Title,Domain,Skill Match (%),Semantic Similarity (%),Hybrid Score (%)
0,Machine Learning Intern,AI/ML,100.00,13.55,65.42
1,Data Analyst,Data Analytics,100.00,6.10,62.44
2,Product Data Analyst,Data Analytics,100.00,5.84,62.34
3,Applied Data Scientist,Data Science,85.71,14.63,57.28
4,Predictive Analytics Scientist,Data Science,83.33,12.12,54.85
5,Data Science Intern,Data Science,77.78,13.02,51.88
6,Junior Data Scientist,Data Science,77.78,12.12,51.52
7,Backend Developer,Web Development,75.00,13.80,50.52
8,Machine Learning Engineer,AI/ML,80.00,5.39,50.16
9,Data Scientist,Data Science,75.00,12.12,49.85


In [102]:
## display the best job
best_job = ranked_jobs.iloc[0]

print("Best Job Recommendation")
print("-----------------------")
print("Job Title:", best_job["job_title"])
print("Domain:", best_job["domain"])
print("Skill Match:", best_job["skill_match_score"], "%")
print("Semantic Similarity:", best_job["semantic_score"], "%")
print("Hybrid Score:", best_job["hybrid_score"], "%")

Best Job Recommendation
-----------------------
Job Title: Machine Learning Intern
Domain: AI/ML
Skill Match: 100.0 %
Semantic Similarity: 13.55 %
Hybrid Score: 65.42 %


In [107]:
## generate skill gap
best_job_skills = best_processed_job["extracted_skills"]

best_job_skill_gap = compare_skills(
    resume_skills,
    best_job_skills
)

print("Best Job:", best_job["job_title"])

print("\nMatched Skills:")
print(best_job_skill_gap["matched_skills"])

print("\nMissing Skills:")
print(best_job_skill_gap["missing_skills"])

print("\nExtra Skills:")
print(best_job_skill_gap["extra_skills"])

Best Job: Machine Learning Intern

Matched Skills:
['machine learning', 'python', 'scikit-learn', 'sql']

Missing Skills:
[]

Extra Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']


In [104]:
print(best_job.keys())

Index(['job_id', 'job_title', 'domain', 'skill_match_score', 'semantic_score',
       'hybrid_score'],
      dtype='str')


In [105]:
best_job_id = best_job["job_id"]

best_processed_job = next(
    job for job in processed_jobs
    if job["job_id"] == best_job_id
)

print(best_processed_job.keys())

dict_keys(['job_id', 'job_title', 'domain', 'file_path', 'required_skills', 'description', 'clean_description', 'extracted_skills'])


In [106]:
print("Best Job:", best_processed_job["job_title"])

Best Job: Machine Learning Intern


### 27.18 — Build Final Recommendation Summary

The final recommendation combines:

- Recommended career domain
- Best matching job
- Skill match score
- Semantic similarity
- Hybrid compatibility score
- Matched skills
- Missing skills
- Extra skills

This creates a single structured result that can later be connected
to a user interface or Streamlit application.

In [108]:
recommendation = {
    "recommended_domain": best_domain,
    "domain_score": round(best_domain_score, 2),

    "best_job": best_processed_job["job_title"],
    "job_id": best_processed_job["job_id"],

    "skill_match_score": best_job["skill_match_score"],
    "semantic_score": best_job["semantic_score"],
    "hybrid_score": best_job["hybrid_score"],

    "matched_skills": best_job_skill_gap["matched_skills"],
    "missing_skills": best_job_skill_gap["missing_skills"],
    "extra_skills": best_job_skill_gap["extra_skills"]
}

In [109]:
print(recommendation)

{'recommended_domain': 'Data Science', 'domain_score': np.float64(53.08), 'best_job': 'Machine Learning Intern', 'job_id': 'ML001', 'skill_match_score': np.float64(100.0), 'semantic_score': np.float64(13.55), 'hybrid_score': np.float64(65.42), 'matched_skills': ['machine learning', 'python', 'scikit-learn', 'sql'], 'missing_skills': [], 'extra_skills': ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']}


In [110]:
print("=" * 50)
print("       RESUME JOB RECOMMENDATION")
print("=" * 50)

print(f"\nRecommended Domain : {recommendation['recommended_domain']}")
print(f"Domain Score       : {recommendation['domain_score']}%")

print(f"\nBest Job            : {recommendation['best_job']}")
print(f"Skill Match         : {recommendation['skill_match_score']}%")
print(f"Semantic Similarity : {recommendation['semantic_score']}%")
print(f"Hybrid Score        : {recommendation['hybrid_score']}%")

print("\nMatched Skills:")
print(recommendation["matched_skills"])

print("\nMissing Skills:")
print(recommendation["missing_skills"])

print("\nExtra Skills:")
print(recommendation["extra_skills"])

print("=" * 50)

       RESUME JOB RECOMMENDATION

Recommended Domain : Data Science
Domain Score       : 53.08%

Best Job            : Machine Learning Intern
Skill Match         : 100.0%
Semantic Similarity : 13.55%
Hybrid Score        : 65.42%

Matched Skills:
['machine learning', 'python', 'scikit-learn', 'sql']

Missing Skills:
[]

Extra Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']


### 27.19 — Skill Gap Analysis for Top Recommended Jobs

We will calculate the skill gap for each of the top recommended jobs.

For every recommended job, the system will identify:

- Matched skills
- Missing skills
- Extra skills

This allows the user to understand what skills are required for
each recommended career opportunity.

In [111]:
top_5_jobs = ranked_jobs.head(5)

top_job_recommendations = []

for _, ranked_job in top_5_jobs.iterrows():

    job_id = ranked_job["job_id"]

    processed_job = next(
        job for job in processed_jobs
        if job["job_id"] == job_id
    )

    skill_gap = compare_skills(
        resume_skills,
        processed_job["extracted_skills"]
    )

    top_job_recommendations.append({
        "job_id": job_id,
        "job_title": ranked_job["job_title"],
        "domain": ranked_job["domain"],
        "hybrid_score": ranked_job["hybrid_score"],
        "matched_skills": skill_gap["matched_skills"],
        "missing_skills": skill_gap["missing_skills"],
        "extra_skills": skill_gap["extra_skills"]
    })

In [124]:
for job in top_job_recommendations:

    print("\n" + "=" * 50)

    print("Job:", job["job_title"])
    print("Domain:", job["domain"])
    print("Hybrid Score:", job["hybrid_score"], "%")

    print("Matched Skills:", job["matched_skills"])
    print("Missing Skills:", job["missing_skills"])
    print("Extra Skills:", job["extra_skills"])


Job: Machine Learning Intern
Domain: AI/ML
Hybrid Score: 65.42 %
Matched Skills: ['machine learning', 'python', 'scikit-learn', 'sql']
Missing Skills: []
Extra Skills: ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']

Job: Data Analyst
Domain: Data Analytics
Hybrid Score: 62.44 %
Matched Skills: ['pandas', 'python', 'sql']
Missing Skills: []
Extra Skills: ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'scikit-learn']

Job: Product Data Analyst
Domain: Data Analytics
Hybrid Score: 62.34 %
Matched Skills: ['pandas', 'python', 'sql']
Missing Skills: []
Extra Skills: ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'scikit-learn']

Job: Applied Data Scientist
Domain: Data Science
Hybrid Score: 57.28 %
Matched Skills: ['machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql']
Missing Skills: 

### 27.20 — Aggregate Skill Gaps

We will combine the missing skills from the top recommended jobs
and count how frequently each skill appears.

Frequently occurring missing skills represent important areas
for career preparation.

In [112]:
from collections import Counter

missing_skill_counter = Counter()

for job in top_job_recommendations:
    missing_skill_counter.update(job["missing_skills"])

print("Skill Gap Frequency:")
print(missing_skill_counter)

Skill Gap Frequency:
Counter({'statistics': 2})


In [113]:
skill_gap_summary = pd.DataFrame(
    missing_skill_counter.items(),
    columns=["Skill", "Number of Jobs"]
).sort_values(
    by="Number of Jobs",
    ascending=False
).reset_index(drop=True)

display(skill_gap_summary)

,Skill,Number of Jobs
0,statistics,2


### 27.21 — Generate Recommendation Explanations

For each recommended job, we will generate a human-readable
explanation describing why the job was recommended.

The explanation will be based on the skill match score,
semantic similarity, matched skills, and missing skills.

In [114]:
def generate_recommendation_explanation(job, skill_gap):
    
    skill_score = job["skill_match_score"]
    semantic_score = job["semantic_score"]
    hybrid_score = job["hybrid_score"]

    matched = skill_gap["matched_skills"]
    missing = skill_gap["missing_skills"]

    explanation = (
        f"This job received a hybrid compatibility score of "
        f"{hybrid_score:.2f}%. "
        f"The resume matches {skill_score:.2f}% of the required skills "
        f"and has a semantic similarity score of "
        f"{semantic_score:.2f}%."
    )

    if matched:
        explanation += (
            f" Matched skills include: {', '.join(matched)}."
        )

    if missing:
        explanation += (
            f" The main skill gap is: {', '.join(missing)}."
        )
    else:
        explanation += " No missing required skills were identified."

    return explanation

In [115]:
best_explanation = generate_recommendation_explanation(
    best_job,
    best_job_skill_gap
)

print(best_explanation)

This job received a hybrid compatibility score of 65.42%. The resume matches 100.00% of the required skills and has a semantic similarity score of 13.55%. Matched skills include: machine learning, python, scikit-learn, sql. No missing required skills were identified.


### 27.22 — Recommendation Strength

Convert the numerical hybrid score into an easy-to-understand
recommendation strength label.

This makes the recommendation output more interpretable for users.

In [116]:
def get_match_strength(score):

    if score >= 80:
        return "Excellent Match"

    elif score >= 65:
        return "Strong Match"

    elif score >= 50:
        return "Moderate Match"

    else:
        return "Weak Match"

In [117]:
print (best_job["hybrid_score"])
print(get_match_strength(best_job["hybrid_score"]))

65.42
Strong Match


### 27.23 — Add Match Strength to Job Recommendations

Convert each job's hybrid compatibility score into an interpretable
match-strength label such as Excellent Match, Strong Match,
Moderate Match, or Weak Match.

In [118]:
for job in top_job_recommendations:

    job["match_strength"] = get_match_strength(
        job["hybrid_score"]
    )

In [134]:
for job in top_job_recommendations:

    print("=" * 50)
    print("Job:", job["job_title"])
    print("Domain:", job["domain"])
    print("Hybrid Score:", job["hybrid_score"], "%")
    print("Match Strength:", job["match_strength"])

Job: Machine Learning Intern
Domain: AI/ML
Hybrid Score: 65.42 %
Match Strength: Strong Match
Job: Data Analyst
Domain: Data Analytics
Hybrid Score: 62.44 %
Match Strength: Moderate Match
Job: Product Data Analyst
Domain: Data Analytics
Hybrid Score: 62.34 %
Match Strength: Moderate Match
Job: Applied Data Scientist
Domain: Data Science
Hybrid Score: 57.28 %
Match Strength: Moderate Match
Job: Predictive Analytics Scientist
Domain: Data Science
Hybrid Score: 54.85 %
Match Strength: Moderate Match


In [119]:
## add match strength to the recommendation table
recommendation_df = pd.DataFrame(
    top_job_recommendations
)

display(
    recommendation_df[
        [
            "job_title",
            "domain",
            "hybrid_score",
            "match_strength",
            "matched_skills",
            "missing_skills"
        ]
    ]
)

,job_title,domain,hybrid_score,match_strength,matched_skills,missing_skills
0,Machine Learning Intern,AI/ML,65.42,Strong Match,"[machine learning, python, scikit-learn, sql]",[]
1,Data Analyst,Data Analytics,62.44,Moderate Match,"[pandas, python, sql]",[]
2,Product Data Analyst,Data Analytics,62.34,Moderate Match,"[pandas, python, sql]",[]
3,Applied Data Scientist,Data Science,57.28,Moderate Match,"[machine learning, numpy, pandas, python, scik...",[statistics]
4,Predictive Analytics Scientist,Data Science,54.85,Moderate Match,"[machine learning, pandas, python, scikit-lear...",[statistics]


### 27.24 — Build the Final Recommendation Function

Combine the job scoring, ranking, domain analysis, skill-gap analysis,
and match-strength calculation into a reusable recommendation function.

This function will serve as the main recommendation pipeline that can
later be connected to the application interface.

In [120]:
def generate_recommendations(
    resume_skills,
    resume_text,
    processed_jobs,
    top_n=5
):
    
    results = []

    # Step 1: Score every job
    for job in processed_jobs:

        score = score_job(
            resume_skills,
            resume_text,
            job
        )

        results.append({
            "job_id": job["job_id"],
            "job_title": job["job_title"],
            "domain": job["domain"],
            "skill_match_score": score["skill_match_score"],
            "semantic_score": score["semantic_score"],
            "hybrid_score": score["hybrid_score"]
        })

    # Step 2: Create DataFrame
    results_df = pd.DataFrame(results)

    # Step 3: Rank jobs
    ranked_jobs = (
        results_df
        .sort_values(
            by="hybrid_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    # Step 4: Domain scores
    domain_scores = (
        results_df
        .groupby("domain")["hybrid_score"]
        .mean()
        .sort_values(ascending=False)
    )

    # Step 5: Select top jobs
    top_jobs = ranked_jobs.head(top_n)

    recommendations = []

    # Step 6: Skill-gap analysis
    for _, ranked_job in top_jobs.iterrows():

        job_id = ranked_job["job_id"]

        processed_job = next(
            job for job in processed_jobs
            if job["job_id"] == job_id
        )

        skill_gap = compare_skills(
            resume_skills,
            processed_job["extracted_skills"]
        )

        recommendations.append({
            "job_id": job_id,
            "job_title": ranked_job["job_title"],
            "domain": ranked_job["domain"],
            "skill_match_score": ranked_job["skill_match_score"],
            "semantic_score": ranked_job["semantic_score"],
            "hybrid_score": ranked_job["hybrid_score"],
            "match_strength": get_match_strength(
                ranked_job["hybrid_score"]
            ),
            "matched_skills": skill_gap["matched_skills"],
            "missing_skills": skill_gap["missing_skills"],
            "extra_skills": skill_gap["extra_skills"]
        })

    # Step 7: Best domain
    best_domain = domain_scores.index[0]

    best_domain_score = domain_scores.iloc[0]

    return {
        "recommended_domain": best_domain,
        "domain_score": best_domain_score,
        "domain_scores": domain_scores,
        "recommendations": recommendations,
        "ranked_jobs": ranked_jobs
    }

In [121]:
## one function call to  run whole recommendation process
final_results = generate_recommendations(
    resume_skills,
    cleaned_resume,
    processed_jobs,
    top_n=5
)

In [122]:
print("Recommended Domain:")
print(final_results["recommended_domain"])

print("\nDomain Score:")
print(
    round(
        final_results["domain_score"],
        2
    ),
    "%"
)

Recommended Domain:
Data Science

Domain Score:
53.08 %


In [123]:
## check top 5
for job in final_results["recommendations"]:

    print("=" * 50)

    print("Job:", job["job_title"])
    print("Domain:", job["domain"])
    print("Hybrid Score:", job["hybrid_score"], "%")
    print("Match Strength:", job["match_strength"])

    print("Missing Skills:")
    print(job["missing_skills"])

Job: Machine Learning Intern
Domain: AI/ML
Hybrid Score: 65.42 %
Match Strength: Strong Match
Missing Skills:
[]
Job: Data Analyst
Domain: Data Analytics
Hybrid Score: 62.44 %
Match Strength: Moderate Match
Missing Skills:
[]
Job: Product Data Analyst
Domain: Data Analytics
Hybrid Score: 62.34 %
Match Strength: Moderate Match
Missing Skills:
[]
Job: Applied Data Scientist
Domain: Data Science
Hybrid Score: 57.28 %
Match Strength: Moderate Match
Missing Skills:
['statistics']
Job: Predictive Analytics Scientist
Domain: Data Science
Hybrid Score: 54.85 %
Match Strength: Moderate Match
Missing Skills:
['statistics']


### 27.25 — Validate Recommendation Pipeline

Validate the output of the recommendation engine before integrating
it with the application.

We will check:

1. Recommended domain
2. Domain scores
3. Number of recommendations
4. Required fields
5. Ranking order

In [124]:
print("Recommended Domain:")
print(final_results["recommended_domain"])

print("\nNumber of Recommendations:")
print(len(final_results["recommendations"]))

print("\nDomain Scores:")
print(final_results["domain_scores"])

Recommended Domain:
Data Science

Number of Recommendations:
5

Domain Scores:
domain
Data Science       53.076
Data Analytics     48.296
AI/ML              45.474
Research           40.856
Web Development    37.978
Name: hybrid_score, dtype: float64


In [125]:
required_fields = [
    "job_id",
    "job_title",
    "domain",
    "skill_match_score",
    "semantic_score",
    "hybrid_score",
    "match_strength",
    "matched_skills",
    "missing_skills",
    "extra_skills"
]

for job in final_results["recommendations"]:

    missing_fields = [
        field
        for field in required_fields
        if field not in job
    ]

    print(
        job["job_title"],
        "→",
        "OK" if not missing_fields else missing_fields
    )

Machine Learning Intern → OK
Data Analyst → OK
Product Data Analyst → OK
Applied Data Scientist → OK
Predictive Analytics Scientist → OK


In [126]:
## verify ranking
scores = [
    job["hybrid_score"]
    for job in final_results["recommendations"]
]

print("Scores:")
print(scores)

print("\nCorrectly Ranked:")
print(scores == sorted(scores, reverse=True))

Scores:
[65.42, 62.44, 62.34, 57.28, 54.85]

Correctly Ranked:
True


### 27.26 — Prepare Application-Ready Recommendation Output

Separate the internal recommendation calculations from the information
that will be displayed to the user.

The application will receive only the relevant recommendation data.

In [127]:
def prepare_recommendation_output(final_results):

    output = {
        "recommended_domain": final_results["recommended_domain"],
        "domain_score": round(
            final_results["domain_score"],
            2
        ),
        "recommendations": []
    }

    for job in final_results["recommendations"]:

        output["recommendations"].append({
            "job_title": job["job_title"],
            "domain": job["domain"],
            "skill_match_score": round(
                job["skill_match_score"],
                2
            ),
            "semantic_score": round(
                job["semantic_score"],
                2
            ),
            "hybrid_score": round(
                job["hybrid_score"],
                2
            ),
            "match_strength": job["match_strength"],
            "matched_skills": job["matched_skills"],
            "missing_skills": job["missing_skills"],
            "extra_skills": job["extra_skills"]
        })

    return output

In [128]:
app_output = prepare_recommendation_output(
     final_results
)

print(app_output)

{'recommended_domain': 'Data Science', 'domain_score': np.float64(53.08), 'recommendations': [{'job_title': 'Machine Learning Intern', 'domain': 'AI/ML', 'skill_match_score': 100.0, 'semantic_score': 13.55, 'hybrid_score': 65.42, 'match_strength': 'Strong Match', 'matched_skills': ['machine learning', 'python', 'scikit-learn', 'sql'], 'missing_skills': [], 'extra_skills': ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']}, {'job_title': 'Data Analyst', 'domain': 'Data Analytics', 'skill_match_score': 100.0, 'semantic_score': 6.1, 'hybrid_score': 62.44, 'match_strength': 'Moderate Match', 'matched_skills': ['pandas', 'python', 'sql'], 'missing_skills': [], 'extra_skills': ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'scikit-learn']}, {'job_title': 'Product Data Analyst', 'domain': 'Data Analytics', 'skill_match_score': 100.0, 'semantic_score': 5.84, 'hybrid_score': 

In [129]:
#check recommendation output
print(
    "Recommended Domain:",
    app_output["recommended_domain"]
)

print(
    "Domain Score:",
    app_output["domain_score"],
    "%"
)

Recommended Domain: Data Science
Domain Score: 53.08 %


In [130]:
#check the domain
print(
    "Recommended Domain:",
    app_output["recommended_domain"]
)

print(
    "Domain Score:",
    app_output["domain_score"],
    "%"
)

Recommended Domain: Data Science
Domain Score: 53.08 %


In [132]:
#check recommendation
for job in app_output["recommendations"]:

    print(
        job["job_title"],
        "|",
        job["hybrid_score"],
        "|",
        job["match_strength"]
    )

Machine Learning Intern | 65.42 | Strong Match
Data Analyst | 62.44 | Moderate Match
Product Data Analyst | 62.34 | Moderate Match
Applied Data Scientist | 57.28 | Moderate Match
Predictive Analytics Scientist | 54.85 | Moderate Match


### 27.27 — Create the End-to-End Recommendation Pipeline

Combine the recommendation engine and output preparation into one
function.

The final application will be able to provide a resume and receive
a clean recommendation result without needing to know the internal
processing steps.

In [133]:
def run_recommendation_pipeline(
    resume_skills,
    resume_text,
    processed_jobs,
    top_n=5
):

    final_results = generate_recommendations(
        resume_skills,
        resume_text,
        processed_jobs,
        top_n=top_n
    )

    app_output = prepare_recommendation_output(
        final_results
    )

    return app_output

In [134]:
app_result = run_recommendation_pipeline(
    resume_skills,
    cleaned_resume,
    processed_jobs,
    top_n=5
)

In [135]:
for job in app_result["recommendations"]:

    print(
        job["job_title"],
        "|",
        job["hybrid_score"],
        "|",
        job["match_strength"]
    )

Machine Learning Intern | 65.42 | Strong Match
Data Analyst | 62.44 | Moderate Match
Product Data Analyst | 62.34 | Moderate Match
Applied Data Scientist | 57.28 | Moderate Match
Predictive Analytics Scientist | 54.85 | Moderate Match


### 27.28 — Create the Recommendation Engine Module

Move the final recommendation pipeline into a dedicated Python module.

This separates the recommendation logic from the notebook and allows
the same pipeline to be reused by the Streamlit application.

### 27.29 — Test Recommendation Engine Module

Import the recommendation pipeline from the source module and verify
that it produces the same recommendation results as the notebook version.

In [136]:
import src.match_score as ms

print(dir(ms))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_hybrid_score']


In [137]:
import inspect

print(inspect.signature(ms.calculate_hybrid_score))

(skill_match_score, semantic_similarity_score, skill_weight=0.6, semantic_weight=0.4)


In [138]:
import src.match_score as ms
import src.semantic_matching as sem
import src.skill_matching as sm

print("MATCH SCORE:")
print(dir(ms))

print("\nSEMANTIC:")
print(dir(sem))

print("\nSKILL MATCHING:")
print(dir(sm))

MATCH SCORE:
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_hybrid_score']

SEMANTIC:
['TfidfVectorizer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_semantic_similarity', 'cosine_similarity']

SKILL MATCHING:
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_skill_match_percentage', 'compare_skills']


In [139]:
import inspect

print("compare_skills:",
      inspect.signature(sm.compare_skills))

print("calculate_skill_match_percentage:",
      inspect.signature(sm.calculate_skill_match_percentage))

print("calculate_semantic_similarity:",
      inspect.signature(sem.calculate_semantic_similarity))

print("calculate_hybrid_score:",
      inspect.signature(ms.calculate_hybrid_score))

compare_skills: (resume_skills, job_skills)
calculate_skill_match_percentage: (matched_skills, job_skills)
calculate_semantic_similarity: (resume_text, job_text)
calculate_hybrid_score: (skill_match_score, semantic_similarity_score, skill_weight=0.6, semantic_weight=0.4)


In [140]:
from src.recommendation_engine import run_recommendation_pipeline

### 27.30 — Test Recommendation Count

Verify that the recommendation engine can dynamically return a requested
number of top jobs using the `top_n` parameter.

In [141]:
import importlib
import src.recommendation_engine as recommendation_engine

importlib.reload(recommendation_engine)

<module 'src.recommendation_engine' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\recommendation_engine.py'>

In [142]:
print(
    hasattr(
        recommendation_engine,
        "run_recommendation_pipeline"
    )
)

True


In [143]:
test_result = recommendation_engine.run_recommendation_pipeline(
    resume_skills,
    cleaned_resume,
    processed_jobs,
    top_n=3
)

In [145]:
print("Number of recommendations:")
print(len(test_result["recommendations"]))

Number of recommendations:
3


### 27.31 — Test Top 10 Recommendations

Verify that the recommendation pipeline can return the requested
number of top-ranked jobs when `top_n=10`.

In [146]:
top10_result = recommendation_engine.run_recommendation_pipeline(
    resume_skills,
    cleaned_resume,
    processed_jobs,
    top_n=10
)

In [147]:
print(
    "Number of recommendations:",
    len(top10_result["recommendations"])
)

Number of recommendations: 10


In [148]:
for i, job in enumerate(
    top10_result["recommendations"],
    start=1
):

    print(
        f"{i}. {job['job_title']} "
        f"| {job['domain']} "
        f"| {job['hybrid_score']}% "
        f"| {job['match_strength']}"
    )

1. Machine Learning Intern | AI/ML | 65.42% | Strong Match
2. Data Analyst | Data Analytics | 62.44% | Moderate Match
3. Product Data Analyst | Data Analytics | 62.34% | Moderate Match
4. Applied Data Scientist | Data Science | 57.28% | Moderate Match
5. Predictive Analytics Scientist | Data Science | 54.85% | Moderate Match
6. Data Science Intern | Data Science | 51.88% | Moderate Match
7. Junior Data Scientist | Data Science | 51.52% | Moderate Match
8. Backend Developer | Web Development | 50.52% | Moderate Match
9. Machine Learning Engineer | AI/ML | 50.16% | Moderate Match
10. Data Scientist | Data Science | 49.85% | Weak Match


### 27.32 — Resume Processing Pipeline

Create a reusable pipeline that converts raw resume text into the
cleaned resume text and extracted skills required by the
recommendation engine.

In [149]:
print("Resume processing functions:")

print(extract_resume_text)
print(clean_resume_text)
print(extract_skills)

Resume processing functions:
<function extract_resume_text at 0x000002EDC10652D0>
deepratim ghosh kolkata india 9163800261 linkedin.com in deepratim-ghosh github.com dpro-123 professional summary computer science engineering undergraduate with an 8.92 cgpa and hands-on experience in machine learning data analytics python sql and practical ml workflows. built end-to-end ml projects involving eda feature engineering model evaluation hyperparameter tuning model interpretability and deployment. currently pursuing a training-cum-industrial internship in ai ml and data analytics seeking internship opportunities in machine learning data science ai or data analytics. education bachelor of engineering b.e. computer science engineering narula institute of technology expected 2028 cgpa 8.92 experience euphoria genx training-cum-industrial intern ai ml data analytics hybrid aug 2026 present working on practical applications of ai ml and data analytics. applying python data preprocessing explorator

In [168]:
import inspect

print("\nSignatures:")

print("extract_resume_text:",
      inspect.signature(extract_resume_text))

print("clean_resume_text:",
      inspect.signature(clean_resume_text))

print("extract_skills:",
      inspect.signature(extract_skills))


Signatures:
extract_resume_text: (file_path)


TypeError: 'deepratim ghosh kolkata india 9163800261 linkedin.com in deepratim-ghosh github.com dpro-123 professional summary computer science engineering undergraduate with an 8.92 cgpa and hands-on experience in machine learning data analytics python sql and practical ml workflows. built end-to-end ml projects involving eda feature engineering model evaluation hyperparameter tuning model interpretability and deployment. currently pursuing a training-cum-industrial internship in ai ml and data analytics seeking internship opportunities in machine learning data science ai or data analytics. education bachelor of engineering b.e. computer science engineering narula institute of technology expected 2028 cgpa 8.92 experience euphoria genx training-cum-industrial intern ai ml data analytics hybrid aug 2026 present working on practical applications of ai ml and data analytics. applying python data preprocessing exploratory analysis and machine learning concepts in an industry-oriented train

In [154]:
print(type(clean_resume_text))

<class 'str'>


In [153]:
import sys

for name, module in list(sys.modules.items()):

    if module is not None and hasattr(module, "clean_resume_text"):

        print(name)

__main__
__mp_main__


In [156]:
import inspect

for name, obj in list(globals().items()):
    if callable(obj):
        if any(word in name.lower() for word in ["resume", "clean", "preprocess", "text"]):
            try:
                print(name, ":", inspect.signature(obj))
            except (TypeError, ValueError):
                print(name, ": signature unavailable")

preprocess_text : (text)
extract_resume_text : (file_path)


### 27.32 — Resume Processing Pipeline

Create a reusable function that takes raw resume text,
preprocesses it, and extracts the skills required by the
job recommendation engine.

In [157]:
def process_resume(resume_text):

    cleaned_text = preprocess_text(
        resume_text
    )

    resume_skills = extract_skills(
        cleaned_text
    )

    return {
        "resume_text": cleaned_text,
        "resume_skills": resume_skills
    }

In [158]:
processed_resume = process_resume(
    raw_resume_text
)

In [159]:
print("Resume Text:")
print(
    processed_resume["resume_text"][:500]
)

print("\nExtracted Skills:")
print(
    processed_resume["resume_skills"]
)

Resume Text:
deepratim ghosh kolkata india 9163800261 linkedin.com in deepratim-ghosh github.com dpro-123 professional summary computer science engineering undergraduate with an 8.92 cgpa and hands-on experience in machine learning data analytics python sql and practical ml workflows. built end-to-end ml projects involving eda feature engineering model evaluation hyperparameter tuning model interpretability and deployment. currently pursuing a training-cum-industrial internship in ai ml and data analytics se

Extracted Skills:
['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql']


### 27.33 — Complete Resume-to-Recommendation Pipeline

Combine resume processing and job recommendation into a single
pipeline.

The pipeline will accept raw resume text and return the final
job recommendations and domain recommendation.

In [160]:
def recommend_from_resume(
    resume_text,
    processed_jobs,
    top_n=5
):

    processed_resume = process_resume(
        resume_text
    )

    result = run_recommendation_pipeline(
        processed_resume["resume_skills"],
        processed_resume["resume_text"],
        processed_jobs,
        top_n=top_n
    )

    return result

In [161]:
complete_result = recommendation_engine.recommend_from_resume(
    raw_resume_text,
    processed_jobs,
    top_n=5
)

AttributeError: module 'src.recommendation_engine' has no attribute 'recommend_from_resume'

In [162]:
import sys

for name, module in list(sys.modules.items()):

    if module is not None:

        if hasattr(module, "preprocess_text") or hasattr(module, "extract_skills"):

            print(name)

__main__
__mp_main__
src.preprocessing
src.skill_extraction
src.job_processing


In [163]:
import src.preprocessing as preprocessing
import src.skill_extraction as skill_extraction
import src.job_processing as job_processing

print("PREPROCESSING:")
print(dir(preprocessing))

print("\nSKILL EXTRACTION:")
print(dir(skill_extraction))

print("\nJOB PROCESSING:")
print(dir(job_processing))

PREPROCESSING:
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'preprocess_text', 're']

SKILL EXTRACTION:
['SHORT_SKILLS', 'SKILL_ALIASES', 'TECHNICAL_SKILLS', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'extract_skills', 're']

JOB PROCESSING:
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'extract_skills', 'preprocess_text', 'process_all_jobs', 'process_job']


In [164]:
import importlib
import src.recommendation_engine as recommendation_engine

importlib.reload(recommendation_engine)

<module 'src.recommendation_engine' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\recommendation_engine.py'>

In [165]:
print(
    hasattr(
        recommendation_engine,
        "recommend_from_resume"
    )
)

True


In [166]:
#test the complete pipeline
complete_result = recommendation_engine.recommend_from_resume(
    raw_resume_text,
    processed_jobs,
    top_n=5
)

In [167]:
print(
    "Recommended Domain:",
    complete_result["recommended_domain"]
)

print("\nTop Recommendations:")

for job in complete_result["recommendations"]:

    print(
        job["job_title"],
        "|",
        job["hybrid_score"],
        "%",
        "|",
        job["match_strength"]
    )

Recommended Domain: Data Science

Top Recommendations:
Machine Learning Intern | 79.22 % | Strong Match
Data Analyst | 71.06 % | Strong Match
Product Data Analyst | 70.61 % | Strong Match
Applied Data Scientist | 66.92 % | Strong Match
Machine Learning Engineer | 65.06 % | Strong Match


### checking why score chnages

In [169]:
new_processed_resume = recommendation_engine.process_resume(
    raw_resume_text
)

print(
    cleaned_resume == new_processed_resume["resume_text"]
)

False


In [170]:
print("Old length:", len(cleaned_resume))
print(
    "New length:",
    len(new_processed_resume["resume_text"])
)

Old length: 97
New length: 2531


In [171]:
#find the first difference
old_text = cleaned_resume
new_text = new_processed_resume["resume_text"]

for i, (a, b) in enumerate(
    zip(old_text, new_text)
):

    if a != b:
        print("First difference at:", i)
        print("Old:", repr(old_text[i:i+100]))
        print("New:", repr(new_text[i:i+100]))
        break

First difference at: 0
Old: 'john doe python developer email github skills python c++ c# sql pandas numpy scikit-learn node.js'
New: 'deepratim ghosh kolkata india 9163800261 linkedin.com in deepratim-ghosh github.com dpro-123 profess'


In [172]:
print("OLD:")
print(cleaned_resume[:500])

print("\nNEW:")
print(new_processed_resume["resume_text"][:500])

OLD:
john doe python developer email github skills python c++ c# sql pandas numpy scikit-learn node.js

NEW:
deepratim ghosh kolkata india 9163800261 linkedin.com in deepratim-ghosh github.com dpro-123 professional summary computer science engineering undergraduate with an 8.92 cgpa and hands-on experience in machine learning data analytics python sql and practical ml workflows. built end-to-end ml projects involving eda feature engineering model evaluation hyperparameter tuning model interpretability and deployment. currently pursuing a training-cum-industrial internship in ai ml and data analytics se


### 27.34 — Resume File Recommendation Pipeline

Create a reusable pipeline that accepts a resume file path,
extracts the resume text, processes the resume, and generates
job recommendations.

In [173]:
import sys

for name, module in list(sys.modules.items()):

    if module is not None and hasattr(
        module,
        "extract_resume_text"
    ):
        print(name)

__main__
__mp_main__
src.resume_parser


In [174]:
import importlib

importlib.reload(recommendation_engine)

<module 'src.recommendation_engine' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\recommendation_engine.py'>

In [175]:
print(
    hasattr(
        recommendation_engine,
        "recommend_from_resume_file"
    )
)

True


In [176]:
#find resume path
import os

for root, dirs, files in os.walk("../"):
    for file in files:
        if file.lower().endswith(".pdf"):
            print(os.path.join(root, file))

../data\raw\resumes\deepratim_ghosh_resume_2026_aug.pdf


In [ ]:
#set the path
resume_path = "../data/raw/resumes/deepratim_ghosh_resume_2026_aug.pdf"

In [178]:
file_result = recommendation_engine.recommend_from_resume_file(
    resume_path,
    processed_jobs,
    top_n=5
)

In [179]:
print("Recommended Domain:", file_result["recommended_domain"])

print("\nTop Recommendations:")

for i, job in enumerate(
    file_result["recommendations"],
    start=1
):
    print(
        f"{i}. {job['job_title']} | "
        f"{job['domain']} | "
        f"{job['hybrid_score']}% | "
        f"{job['match_strength']}"
    )

Recommended Domain: Data Science

Top Recommendations:
1. Machine Learning Intern | AI/ML | 79.22% | Strong Match
2. Data Analyst | Data Analytics | 71.06% | Strong Match
3. Product Data Analyst | Data Analytics | 70.61% | Strong Match
4. Applied Data Scientist | Data Science | 66.92% | Strong Match
5. Machine Learning Engineer | AI/ML | 65.06% | Strong Match


### 27.35 — Inspect Final Recommendation Output

Inspect the complete output returned by the recommendation engine
before connecting it to the application interface.

In [180]:
print(file_result.keys())

dict_keys(['recommended_domain', 'domain_score', 'recommendations'])


In [181]:
for key, value in file_result.items():

    print("\n" + "=" * 50)
    print(key)
    print("=" * 50)

    print(value)


recommended_domain
Data Science

domain_score
62.56

recommendations
[{'job_title': 'Machine Learning Intern', 'domain': 'AI/ML', 'skill_match_score': 100.0, 'semantic_score': 48.05, 'hybrid_score': 79.22, 'match_strength': 'Strong Match', 'matched_skills': ['machine learning', 'python', 'scikit-learn', 'sql'], 'missing_skills': [], 'extra_skills': ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']}, {'job_title': 'Data Analyst', 'domain': 'Data Analytics', 'skill_match_score': 100.0, 'semantic_score': 27.65, 'hybrid_score': 71.06, 'match_strength': 'Strong Match', 'matched_skills': ['pandas', 'python', 'sql'], 'missing_skills': [], 'extra_skills': ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'machine learning', 'numpy', 'scikit-learn']}, {'job_title': 'Product Data Analyst', 'domain': 'Data Analytics', 'skill_match_score': 100.0, 'semantic_score': 26.52, 'hybrid_score': 70.61, 'match_strength

### 27.35 — Prepare Results for the Application

Create a presentation-ready result structure containing the
recommended domain, domain score, and ranked job recommendations.

In [182]:
importlib.reload(recommendation_engine)

<module 'src.recommendation_engine' from 'd:\\Deepratim\\resume_screening_job_recommendation\\src\\recommendation_engine.py'>

In [183]:
formatted_result = recommendation_engine.format_recommendation_results(
    file_result
)

In [184]:
print(formatted_result["Recommended Domain"])
print(formatted_result["Domain Score"])

print("\nTop Job:")
print(formatted_result["Recommendations"][0])

Data Science
62.56

Top Job:
{'Job Title': 'Machine Learning Intern', 'Domain': 'AI/ML', 'Skill Match': 100.0, 'Semantic Score': 48.05, 'Hybrid Score': 79.22, 'Match Strength': 'Strong Match', 'Matched Skills': ['machine learning', 'python', 'scikit-learn', 'sql'], 'Missing Skills': [], 'Extra Skills': ['c', 'c++', 'data science', 'deep learning', 'git', 'github', 'java', 'javascript', 'numpy', 'pandas']}


In [185]:
from src.job_processing import process_all_jobs

test_jobs = process_all_jobs(
    "../data/raw/job_descriptions"
)

print(type(test_jobs))

TypeError: string indices must be integers, not 'str'

In [186]:
if isinstance(test_jobs, list):
    print("Length:", len(test_jobs))
    print("First item type:", type(test_jobs[0]))
    print("First item:", test_jobs[0])

elif isinstance(test_jobs, dict):
    print("Keys:", test_jobs.keys())
    print("First value type:", type(next(iter(test_jobs.values()))))
    print("First value:", next(iter(test_jobs.values())))

NameError: name 'test_jobs' is not defined

In [187]:
import inspect

print(
    inspect.signature(process_all_jobs)
)

(jobs)


In [188]:
print(
    inspect.getsource(process_all_jobs)
)

def process_all_jobs(jobs):
    """
    Process every job in the job dataset.
    """

    processed_jobs = []

    for job in jobs:
        processed_job = process_job(job)
        processed_jobs.append(processed_job)

    return processed_jobs



In [189]:
print(type(jobs_df))
print(jobs_df.columns.tolist())
print(jobs_df.shape)

<class 'pandas.DataFrame'>
['job_id', 'job_title', 'domain', 'file_path', 'required_skills']
(25, 5)


In [190]:
print(type(jobs_df))
print(jobs_df.columns.tolist())
print(jobs_df.shape)

<class 'pandas.DataFrame'>
['job_id', 'job_title', 'domain', 'file_path', 'required_skills']
(25, 5)


### create jobs.csv

In [191]:
jobs_df.to_csv(
    "../data/raw/jobs.csv",
    index=False
)

In [192]:
import os

print(
    os.path.exists("../data/raw/jobs.csv")
)

True


In [193]:
print(
    pd.read_csv("../data/raw/jobs.csv").shape
)

(25, 5)
